In [ ]:
# =============================================================================
# Cell 1: Read S2S/ERA5 data and write table_stage_member_raw_factors_SHF.csv
# =============================================================================
import os
import glob
import warnings
import numpy as np
import pandas as pd
import xarray as xr

warnings.filterwarnings("ignore", category=FutureWarning)

print("\n" + "=" * 80)
print("Cell 1: build raw SHF factor table from S2S/ERA5 inputs...")
print("=" * 80)

EXP_OUT_DIR = "/data1/huangy/fig6/NC/v6"
RESID_OLS_DIR = f"{EXP_OUT_DIR}/stage_specific_residualized_ols_SHF_0608_0612_v2"
SUB_DIRS = {
    "tables": f"{RESID_OLS_DIR}/tables",
    "corr_raw": f"{RESID_OLS_DIR}/figures/corr_raw",
    "corr_resid": f"{RESID_OLS_DIR}/figures/corr_resid",
    "lmg_main": f"{RESID_OLS_DIR}/figures/lmg_main",
    "lmg_sensitivity": f"{RESID_OLS_DIR}/figures/lmg_sensitivity",
    "ols_lmg_combo": f"{RESID_OLS_DIR}/figures/ols_lmg_combo",
    "tmax_timeseries": f"{RESID_OLS_DIR}/figures/tmax_timeseries",
    "metrics_heatmap": f"{RESID_OLS_DIR}/figures/metrics_heatmap",
}
for d in SUB_DIRS.values():
    os.makedirs(d, exist_ok=True)

start_date_list_run = ["2023-06-08", "2023-06-12"]
study_window_dict = {
    "Stage-I_dry": ("2023-06-14", "2023-06-17"),
    "Stage-II_wet": ("2023-06-18", "2023-06-24"),
    "Total": ("2023-06-14", "2023-06-24"),
}
study_window_dict_run = study_window_dict

DEBUG_MODE = False
factor_cols_raw = [
    "WNPSH", "Initial_MSEstar_max_NCHN", "NCVI", "NCHN_tp",
    "sm_avg", "SSR_Avg", "SHF_Avg", "z500_anom_NCHN", "Initial_Barrier_NCHN",
]

ERA5_TMAX_FILE = "/data1/huangy/fig6/NC/ai_test/pangu/era5_T/era5_2m_temperature_6_2023.nc"
CF_DIR = "/data1/huangy/fig6/NC/cf_2023-6"
PF_DIR = "/data1/huangy/fig6/NC/2023-06"
MSE_DIR = "/data1/huangy/fig6/NC/MSE"
OROG_FILE = f"{MSE_DIR}/ecmf_cf_orog_2023-06.grib"
NCHN_BOX = (35, 44, 114, 119)
WNPSH_BOX = (15, 25, 115, 150)
Z500_REFORECAST_CANDIDATES = [
    "/data1/huangy/fig6/NC/pl_z03_22/merged_{exp_start_date}.nc",
    "/data6/bjlu/s2s/ecmwf/reforecast/pf/2023-06/pl_z03_22/merged_{exp_start_date}.nc",
    f"{MSE_DIR}/Antecedent/merged_{{exp_start_date}}.nc",
    f"{MSE_DIR}/Antecedent/z500/merged_{{exp_start_date}}.nc",
]
EXPECTED_RAW_ROWS = len(start_date_list_run) * len(study_window_dict_run) * 51
EXPECTED_OLS_COMBINATIONS = len(start_date_list_run) * len(study_window_dict_run)

FILE_CACHE = {}

def get_ds(path):
    if path in FILE_CACHE:
        return FILE_CACHE[path]
    ext = str(path).lower()
    if DEBUG_MODE and ext.endswith((".grb", ".grib")):
        print(f"  [DEBUG get_ds] opening GRIB: {path}")
    if not os.path.exists(path):
        print(f"  [Error] 文件不存在: {path}")
        return None
    try:
        if ext.endswith((".grb", ".grib")):
            FILE_CACHE[path] = xr.open_dataset(path, engine="cfgrib", backend_kwargs={"indexpath": ""})
        else:
            FILE_CACHE[path] = xr.open_dataset(path)
    except Exception as e:
        print(f"  [Critical Error] 文件物理损坏或读取失败: {path}\n    具体原因: {e}")
        FILE_CACHE[path] = None
    return FILE_CACHE[path]

def _detect_lat_dim(obj):
    for dim in ["latitude", "lat"]:
        if dim in obj.dims or dim in obj.coords:
            return dim
    return "latitude"

def _detect_lon_dim(obj):
    for dim in ["longitude", "lon"]:
        if dim in obj.dims or dim in obj.coords:
            return dim
    return "longitude"

def _detect_level_dim(obj):
    for dim in ["pressure_level", "level", "isobaricInhPa"]:
        if dim in obj.dims:
            return dim
    return None

def _has_level_dim(obj):
    return _detect_level_dim(obj) is not None

def _get_var(obj, candidates):
    if isinstance(obj, xr.DataArray):
        return obj
    if obj is None:
        raise ValueError("试图从空(None)数据中提取变量。")
    for var in candidates:
        if var in obj.data_vars:
            return obj[var]
    return obj[list(obj.data_vars)[0]]

def load_s2s_ensemble_with_cf(pf_path, cf_path, var_candidates, level=None):
    pf_ds = get_ds(pf_path)
    cf_ds = get_ds(cf_path)
    if pf_ds is None or cf_ds is None:
        return None
    pf_da = _get_var(pf_ds, var_candidates)
    cf_da = _get_var(cf_ds, var_candidates)
    if level is not None:
        if _has_level_dim(pf_da):
            pf_da = pf_da.sel({_detect_level_dim(pf_da): level}, method="nearest")
        if _has_level_dim(cf_da):
            cf_da = cf_da.sel({_detect_level_dim(cf_da): level}, method="nearest")
    if "number" not in cf_da.dims:
        cf_da = cf_da.expand_dims(number=[0])
    else:
        cf_da = cf_da.assign_coords(number=[0])
    try:
        return xr.concat([cf_da, pf_da], dim="number")
    except Exception:
        return pf_da

def safe_slice_region(da, lat_min, lat_max, lon_min, lon_max):
    lat_dim = _detect_lat_dim(da)
    lon_dim = _detect_lon_dim(da)
    lat_slice = slice(lat_max, lat_min) if float(da[lat_dim][0]) > float(da[lat_dim][-1]) else slice(lat_min, lat_max)
    return da.sel({lat_dim: lat_slice, lon_dim: slice(lon_min, lon_max)})

def load_era5_tmax_daily(filepath):
    print(f"  [ERA5 Tmax] file path: {filepath}")
    if not os.path.exists(filepath):
        print("  [ERA5 Tmax] file missing; daily Tmax unavailable.")
        return None
    try:
        ds = xr.open_dataset(filepath)
        da = _get_var(ds, ["t2m", "2t", "2m_temperature"])
        rn = {d: ("latitude" if "lat" in str(d).lower() else "longitude") for d in da.dims if "lat" in str(d).lower() or "lon" in str(d).lower()}
        if rn:
            da = da.rename(rn)
        da = safe_slice_region(da, *NCHN_BOX)
        raw_units = str(da.attrs.get("units", "")).lower().strip()
        finite_vals = da.values[np.isfinite(da.values)]
        raw_median = float(np.nanmedian(finite_vals)) if finite_vals.size > 0 else np.nan
        if raw_units in ["k", "kelvin"] or (not np.isnan(raw_median) and raw_median > 150):
            da = da - 273.15
            da.attrs["units"] = "degC"
        tc = next((c for c in ["time", "valid_time", "date"] if c in da.coords), None)
        if tc is None:
            return None
        if tc not in da.dims:
            da = da.swap_dims({da[tc].dims[0]: tc})
        if tc != "time":
            da = da.rename({tc: "time"})
        da = da.assign_coords(time=pd.to_datetime(da["time"].values) + pd.Timedelta(hours=8))
        print("  [ERA5 Tmax] UTC+8h conversion completed: True")
        da_daily = da.resample(time="1D").max()
        da_daily_mean = da_daily.mean(dim=[_detect_lat_dim(da_daily), _detect_lon_dim(da_daily)])
        daily_mean = float(da_daily_mean.mean().values)
        daily_min = float(da_daily_mean.min().values)
        daily_max = float(da_daily_mean.max().values)
        date_min = pd.to_datetime(da_daily_mean.time.values).min().strftime("%Y-%m-%d")
        date_max = pd.to_datetime(da_daily_mean.time.values).max().strftime("%Y-%m-%d")
        print(f"  [ERA5 Tmax] daily Tmax date range: {date_min} to {date_max}")
        print(f"  [ERA5 Tmax] daily Tmax min/max/mean: {daily_min:.2f}/{daily_max:.2f}/{daily_mean:.2f} °C")
        if daily_mean > 100:
            raise ValueError(f"ERA5 Tmax daily mean > 100 ({daily_mean:.2f}); unit conversion likely failed.")
        return da_daily_mean
    except Exception as e:
        print(f"  [Warning] ERA5 Tmax load failed: {e}")
        return None

def get_s2s_init_slice(ds, var_name, exp_start_date):
    tc = next((c for c in ["time", "valid_time", "forecast_reference_time", "date"] if c in ds.coords), None)
    if not tc:
        return None
    dates_in_ds = pd.to_datetime(ds[tc].values).date
    target_date = pd.Timestamp(exp_start_date).date()
    if target_date not in dates_in_ds:
        return None
    target_idx = np.where(dates_in_ds == target_date)[0][0]
    actual_dim = ds[tc].dims[0]
    da_init = ds[var_name].isel({actual_dim: target_idx})
    rn = {d: ("latitude" if "lat" in str(d).lower() else "longitude") for d in da_init.dims if "lat" in str(d).lower() or "lon" in str(d).lower()}
    if rn:
        da_init = da_init.rename(rn)
    return da_init

def align_to_lead_day(da):
    if da is None:
        return None
    step_hours = pd.to_timedelta(da.step.values).total_seconds() / 3600.0
    lead_float = step_hours / 24.0
    if not np.allclose(lead_float, np.round(lead_float), atol=1e-6):
        print(f"  [Warning] align_to_lead_day: non-24h-multiple steps detected: {step_hours[:10]}")
    lead_days = np.round(lead_float).astype(int)
    da_ld = da.assign_coords(lead_day=("step", lead_days)).swap_dims({"step": "lead_day"})
    if len(da_ld.lead_day) != len(np.unique(da_ld.lead_day)):
        da_ld = da_ld.groupby("lead_day").mean(dim="lead_day")
    valid_lds = [ld for ld in [1, 2, 3] if ld in da_ld.lead_day.values]
    if not valid_lds:
        return None
    return da_ld.sel(lead_day=valid_lds)

def align_common_lead_day(arrays):
    valid_arrays = [a for a in arrays if a is not None]
    if not valid_arrays:
        return [None] * len(arrays)
    common_lds = set(valid_arrays[0].lead_day.values)
    for a in valid_arrays[1:]:
        common_lds = common_lds.intersection(set(a.lead_day.values))
    common_lds = sorted(list(common_lds))
    if not common_lds:
        return [None] * len(arrays)
    return [a.sel(lead_day=common_lds) if a is not None else None for a in arrays]

def process_s2s_tmax_bjt_daily_max(da_init, region=None):
    if da_init is None:
        return None
    tc = next((c for c in ["time", "valid_time", "forecast_reference_time", "date"] if c in da_init.coords), None)
    bjt_time = pd.to_datetime(da_init[tc].values) + da_init.step.values + pd.Timedelta(hours=8)
    da = da_init.assign_coords(valid_bjt=("step", bjt_time))
    if tc in da.coords and tc not in da.dims:
        da = da.drop_vars(tc)
    da = da.swap_dims({"step": "valid_bjt"}).rename({"valid_bjt": "time"}).sortby("time")
    raw_units = str(da.attrs.get("units", "")).lower().strip()
    finite_vals = da.values[np.isfinite(da.values)]
    raw_median = float(np.nanmedian(finite_vals)) if finite_vals.size > 0 else np.nan
    if raw_units in ["k", "kelvin"] or (not np.isnan(raw_median) and raw_median > 150):
        da = da - 273.15
        da.attrs["units"] = "degC"
    da_daily = da.resample(time="1D").max()
    if region:
        da_daily = safe_slice_region(da_daily, *region)
    lat_dim = _detect_lat_dim(da_daily)
    lon_dim = _detect_lon_dim(da_daily)
    da_daily_mean = da_daily.mean(dim=[lat_dim, lon_dim])
    if float(np.nanmean(da_daily_mean.values)) > 100:
        raise ValueError("S2S Tmax still appears to be Kelvin.")
    return da_daily_mean

def _apportion_accum_to_calendar_days(da_init, force_non_negative=False):
    other_dims = [d for d in da_init.dims if d != "time"]
    da = da_init.transpose("time", *other_dims).sortby("time")
    times = pd.to_datetime(da.time.values)
    vals = da.values
    daily_accum = {}
    for i in range(1, len(times)):
        t0, t1 = times[i - 1], times[i]
        inc_val = vals[i] - vals[i - 1]
        if force_non_negative:
            inc_val = np.maximum(inc_val, 0.0)
        total_duration = (t1 - t0).total_seconds()
        if total_duration <= 0:
            continue
        d0, d1 = t0.date(), t1.date()
        if d0 == d1 or (t1.hour == 0 and t1.minute == 0 and t1.second == 0 and t1.microsecond == 0 and d1 == d0 + pd.Timedelta(days=1)):
            date_str = d0.strftime("%Y-%m-%d")
            daily_accum.setdefault(date_str, np.zeros_like(inc_val))
            daily_accum[date_str] += inc_val
        else:
            current_t = t0
            while current_t < t1:
                next_midnight = pd.Timestamp(current_t.date() + pd.Timedelta(days=1))
                segment_end = min(next_midnight, t1)
                segment_duration = (segment_end - current_t).total_seconds()
                ratio = segment_duration / total_duration
                date_str = current_t.date().strftime("%Y-%m-%d")
                daily_accum.setdefault(date_str, np.zeros_like(inc_val))
                daily_accum[date_str] += inc_val * ratio
                current_t = segment_end
    dates = sorted(list(daily_accum.keys()))
    if not dates:
        return xr.DataArray(np.nan, dims=["time"], coords={"time": [pd.Timestamp(times[0].date())]})
    time_coords = pd.to_datetime(dates)
    stacked_vals = np.stack([daily_accum[d] for d in dates], axis=0)
    new_coords = {"time": time_coords}
    for dim in other_dims:
        new_coords[dim] = da.coords[dim]
    return xr.DataArray(stacked_vals, dims=["time"] + other_dims, coords=new_coords)

def process_accum_variable(da_init, var_type, region=None):
    if da_init is None:
        return None
    tc = next((c for c in ["time", "valid_time", "forecast_reference_time", "date"] if c in da_init.coords), None)
    bjt_time = pd.to_datetime(da_init[tc].values) + da_init.step.values + pd.Timedelta(hours=8)
    da_accum = da_init.assign_coords(valid_bjt=("step", bjt_time))
    if tc in da_accum.coords and tc not in da_accum.dims:
        da_accum = da_accum.drop_vars(tc)
    da_accum = da_accum.swap_dims({"step": "valid_bjt"}).rename({"valid_bjt": "time"}).sortby("time")
    if var_type == "precip":
        da_daily = _apportion_accum_to_calendar_days(da_accum, force_non_negative=True)
        if str(da_init.attrs.get("units", "")).lower().strip() in ["m", "meter", "meters"]:
            da_daily = da_daily * 1000.0
    elif var_type == "flux_ssr":
        da_daily = _apportion_accum_to_calendar_days(da_accum, force_non_negative=False) / 86400.0
    elif var_type == "flux_shf":
        da_daily = _apportion_accum_to_calendar_days(da_accum, force_non_negative=False) / 86400.0
        # S2S SHF is downward-positive by default; SHF_Avg is stored as upward-positive.
        da_daily = -1.0 * da_daily
    else:
        raise ValueError(f"Unsupported accumulated variable type: {var_type}")
    if region:
        da_daily = safe_slice_region(da_daily, *region)
    lat_dim = _detect_lat_dim(da_daily)
    lon_dim = _detect_lon_dim(da_daily)
    return da_daily.mean(dim=[lat_dim, lon_dim])

def process_s2s_bjt_daily_mean(da_init, var_type, region=None):
    if da_init is None:
        return None
    tc = next((c for c in ["time", "valid_time", "forecast_reference_time", "date"] if c in da_init.coords), None)
    bjt_time = pd.to_datetime(da_init[tc].values) + da_init.step.values + pd.Timedelta(hours=8)
    da = da_init.assign_coords(valid_bjt=("step", bjt_time))
    if tc in da.coords and tc not in da.dims:
        da = da.drop_vars(tc)
    da = da.swap_dims({"step": "valid_bjt"}).rename({"valid_bjt": "time"}).sortby("time")
    if var_type == "soil_moisture":
        raw_units = str(da.attrs.get("units", "")).lower().replace("**", "^").replace(" ", "")
        if raw_units in ["kgm-3", "kgm^-3", "kg/m3", "kg/m^3"]:
            da = da / 1000.0
    elif var_type in ["z500", "wnpsh"]:
        if float(da.mean()) > 10000:
            da = da / 9.80665
    da_daily = da.resample(time="1D").mean()
    if region:
        da_daily = safe_slice_region(da_daily, *region)
    lat_dim = _detect_lat_dim(da_daily)
    lon_dim = _detect_lon_dim(da_daily)
    return da_daily.mean(dim=[lat_dim, lon_dim])

def extract_window_only(da_daily, w_start, w_end, agg_func="mean"):
    if da_daily is None:
        return None
    ws, we = pd.Timestamp(w_start).date(), pd.Timestamp(w_end).date()
    vd = pd.to_datetime(da_daily.time.values).date
    idx = [i for i, d in enumerate(vd) if ws <= d <= we]
    if not idx:
        return None
    da_win = da_daily.isel(time=idx)
    return da_win.sum(dim="time").values if agg_func == "sum" else da_win.mean(dim="time").values

def load_z500_reforecast_climatology(exp_start_date, region=NCHN_BOX):
    z500_hc_path = None
    for template in Z500_REFORECAST_CANDIDATES:
        path = template.format(exp_start_date=exp_start_date)
        if os.path.exists(path) and z500_hc_path is None:
            z500_hc_path = path
    if z500_hc_path is None:
        raise FileNotFoundError(f"Z500 reforecast missing for {exp_start_date}")
    ds = xr.open_dataset(z500_hc_path)
    da = _get_var(ds, ["gh", "z", "geopotential"])
    lev_dim = _detect_level_dim(da)
    if lev_dim:
        da = da.sel({lev_dim: 500}, method="nearest")
    rn = {}
    for d in da.dims:
        dl = str(d).lower()
        if "lat" in dl and dl != "latitude":
            rn[d] = "latitude"
        elif "lon" in dl and dl != "longitude":
            rn[d] = "longitude"
    if rn:
        da = da.rename(rn)
    if "time" in da.dims:
        da = da.rename({"time": "hc_time"})
    if float(da.mean()) > 10000:
        da = da / 9.80665
        da.attrs["units"] = "gpm"
    bjt_time = pd.Timestamp(exp_start_date) + da.step.values + pd.Timedelta(hours=8)
    da = da.assign_coords(valid_bjt=("step", bjt_time))
    da = da.swap_dims({"step": "valid_bjt"}).rename({"valid_bjt": "time"}).sortby("time")
    da_daily = da.resample(time="1D").mean()
    da_daily = safe_slice_region(da_daily, region[0], region[1], region[2], region[3])
    da_daily = da_daily.mean(dim=[_detect_lat_dim(da_daily), _detect_lon_dim(da_daily)], skipna=True)
    drop_dims = [d for d in da_daily.dims if d != "time"]
    return da_daily.mean(dim=drop_dims, skipna=True) if drop_dims else da_daily

def apply_z500_anomaly(da_z500_rt_daily, z500_clim):
    if da_z500_rt_daily is None or z500_clim is None:
        return None
    rt_aligned, clim_aligned = xr.align(da_z500_rt_daily, z500_clim, join="inner")
    da_z500_anom = rt_aligned - clim_aligned
    if "number" not in da_z500_anom.dims or da_z500_anom.sizes["number"] != 51:
        raise ValueError("Z500 anomaly missing 'number' dim or size != 51.")
    return da_z500_anom

cp, Lv, g, Rd = 1005.0, 2.5e6, 9.81, 287.05

def _qsat(T_K, p_hPa):
    t_c = T_K - 273.15
    es_Pa = 6.112 * np.exp((17.67 * t_c) / (t_c + 243.5)) * 100.0
    return (0.622 * es_Pa) / (p_hPa * 100.0 - 0.378 * es_Pa)

def _lcl_pressure(T2m_K, Td2m_K, sp_Pa):
    lcl_T = 56.0 + 1.0 / (1.0 / (Td2m_K - 56.0) + np.log(T2m_K / Td2m_K) / 800.0)
    return (sp_Pa * (lcl_T / T2m_K) ** (cp / Rd)) / 100.0

def compute_real_ncvi(exp_start_date):
    rt_da_all = load_s2s_ensemble_with_cf(f"{MSE_DIR}/ecmf_pf_60_2023-06.grib", f"{CF_DIR}/ecmf_cf_60_2023-06.grib", ["pv"])
    da_rt = get_s2s_init_slice(rt_da_all.to_dataset(name="pv"), "pv", exp_start_date) if rt_da_all is not None else None
    if da_rt is None:
        return None
    da_rt_mean = safe_slice_region(da_rt, 35, 50, 115, 130).mean(dim=[_detect_lat_dim(da_rt), _detect_lon_dim(da_rt)])
    da_rt_ld = align_to_lead_day(da_rt_mean)
    date_dash = exp_start_date
    date_nodash = exp_start_date.replace("-", "")
    hc_pf_candidates = [
        f"{MSE_DIR}/Antecedent/ecmf_pf_NCVI_pt320_60_{date_dash}.grib",
        f"{MSE_DIR}/Antecedent/ecmf_pf_NCVI_pt320_{date_dash}.grib",
        f"{MSE_DIR}/Antecedent/ecmf_pf_NCVI_60_{date_dash}.grib",
        f"{MSE_DIR}/Antecedent/ecmf_pf_NCVI_{date_dash}.grib",
    ]
    hc_cf_candidates = [
        f"{CF_DIR}/ecmf_cf_NCVI_pt320_60_{date_dash}.grib",
        f"{CF_DIR}/ecmf_cf_NCVI_pt320_{date_dash}.grib",
        f"{CF_DIR}/ecmf_cf_NCVI_60_{date_dash}.grib",
        f"{CF_DIR}/ecmf_cf_NCVI_{date_dash}.grib",
    ]
    hc_pf_candidates += sorted(set(glob.glob(f"{MSE_DIR}/Antecedent/**/*NCVI*{date_dash}*.grib", recursive=True) + glob.glob(f"{MSE_DIR}/Antecedent/**/*NCVI*{date_nodash}*.grib", recursive=True)))
    hc_cf_candidates += sorted(set(glob.glob(f"{CF_DIR}/**/*NCVI*{date_dash}*.grib", recursive=True) + glob.glob(f"{CF_DIR}/**/*NCVI*{date_nodash}*.grib", recursive=True) + glob.glob(f"{CF_DIR}/**/*ncvi*{date_dash}*.grib", recursive=True) + glob.glob(f"{CF_DIR}/**/*ncvi*{date_nodash}*.grib", recursive=True)))
    hc_pf = next((p for p in hc_pf_candidates if os.path.exists(p)), None)
    hc_cf = next((p for p in hc_cf_candidates if os.path.exists(p)), None)
    da_anomaly = da_rt_ld
    if hc_pf and hc_cf:
        hc_da_all = load_s2s_ensemble_with_cf(hc_pf, hc_cf, ["pv"])
        if hc_da_all is not None:
            rn = {d: ("latitude" if "lat" in str(d).lower() else "longitude") for d in hc_da_all.dims if "lat" in str(d).lower() or "lon" in str(d).lower()}
            if rn:
                hc_da_all = hc_da_all.rename(rn)
            da_hc = get_s2s_init_slice(hc_da_all.to_dataset(name="pv"), "pv", exp_start_date)
            if da_hc is None:
                da_hc = hc_da_all
            if da_hc is not None:
                da_hc_mean = safe_slice_region(da_hc, 35, 50, 115, 130).mean(dim=["latitude", "longitude"])
                da_hc_ld = align_to_lead_day(da_hc_mean)
                if da_hc_ld is not None:
                    drop_dims = [d for d in da_hc_ld.dims if d != "lead_day"]
                    da_hc_clim = da_hc_ld.mean(dim=drop_dims, skipna=True) if drop_dims else da_hc_ld
                    aligned = align_common_lead_day([da_rt_ld, da_hc_clim])
                    if aligned[0] is not None:
                        da_anomaly = aligned[0] - aligned[1]
    if da_anomaly is not None:
        ncvi_val = np.asarray(da_anomaly.mean(dim="lead_day").values).squeeze() * 1e6
        if np.shape(ncvi_val) != (51,):
            return None
        return ncvi_val
    return None

def compute_real_mse(exp_start_date):
    t2m_all = load_s2s_ensemble_with_cf(f"{MSE_DIR}/ecmf_pf_sfc_2t_2023-06.grib", f"{CF_DIR}/ecmf_cf_sfc_2t_2023-06.grib", ["2t", "t2m"])
    td2m_all = load_s2s_ensemble_with_cf(f"{MSE_DIR}/ecmf_pf_sfc_2td_2023-06.grib", f"{CF_DIR}/ecmf_cf_sfc_2td_2023-06.grib", ["2d", "d2m"])
    sp_all = load_s2s_ensemble_with_cf(f"{MSE_DIR}/ecmf_pf_sfc_sp_2023-06.grib", f"{CF_DIR}/ecmf_cf_sfc_sp_2023-06.grib", ["sp"])
    t2m = get_s2s_init_slice(t2m_all.to_dataset(name="v"), "v", exp_start_date) if t2m_all is not None else None
    td2m = get_s2s_init_slice(td2m_all.to_dataset(name="v"), "v", exp_start_date) if td2m_all is not None else None
    sp = get_s2s_init_slice(sp_all.to_dataset(name="v"), "v", exp_start_date) if sp_all is not None else None
    if any(da is None for da in [t2m, td2m, sp]):
        return None, None
    t2m = safe_slice_region(t2m, 35, 44, 114, 119)
    td2m = safe_slice_region(td2m, 35, 44, 114, 119)
    sp = safe_slice_region(sp, 35, 44, 114, 119)
    t2m_ld = align_to_lead_day(t2m)
    td2m_ld = align_to_lead_day(td2m)
    sp_ld = align_to_lead_day(sp)
    t2m_ld, td2m_ld, sp_ld = align_common_lead_day([t2m_ld, td2m_ld, sp_ld])
    if any(da is None for da in [t2m_ld, td2m_ld, sp_ld]):
        return None, None
    orog_ds = get_ds(OROG_FILE)
    if orog_ds is None:
        return None, None
    orog_raw = _get_var(orog_ds, ["z", "orog"])
    rn = {}
    for d in orog_raw.dims:
        dl = str(d).lower()
        if "lat" in dl:
            rn[d] = "latitude"
        elif "lon" in dl:
            rn[d] = "longitude"
    if rn:
        orog_raw = orog_raw.rename(rn)
    orog = safe_slice_region(orog_raw, 35, 44, 114, 119)
    target_grid = t2m_ld.isel(number=0, lead_day=0, drop=True)
    try:
        orog = orog.interp(latitude=target_grid.latitude, longitude=target_grid.longitude, method="nearest")
    except Exception:
        orog = orog.reindex_like(target_grid, method="nearest")
    drop_dims = [d for d in orog.dims if d not in ("latitude", "longitude")]
    if drop_dims:
        orog = orog.mean(dim=drop_dims)
    gz_s_grid = xr.where(orog > 9000, orog, g * orog)
    mse_s_grid = cp * t2m_ld + Lv * _qsat(td2m_ld, sp_ld / 100.0) + gz_s_grid
    lcl_p_grid = _lcl_pressure(t2m_ld, td2m_ld, sp_ld)
    mse_star_levs = []
    for lev in [1000, 925, 850, 700, 500, 300]:
        t_all = load_s2s_ensemble_with_cf(f"{MSE_DIR}/ecmf_pl{lev}_130_2023-06.grib", f"{CF_DIR}/ecmf_cf_pl{lev}_130_2023-06.grib", ["t"])
        z_all = load_s2s_ensemble_with_cf(f"{MSE_DIR}/ecmf_pl{lev}_156_2023-06.grib", f"{CF_DIR}/ecmf_cf_pl{lev}_156_2023-06.grib", ["z", "gh"])
        if t_all is None or z_all is None:
            continue
        t_lev = get_s2s_init_slice(t_all.to_dataset(name="v"), "v", exp_start_date)
        z_lev = get_s2s_init_slice(z_all.to_dataset(name="v"), "v", exp_start_date)
        if t_lev is None or z_lev is None:
            continue
        t_lev = safe_slice_region(t_lev, 35, 44, 114, 119)
        z_lev = safe_slice_region(z_lev, 35, 44, 114, 119)
        t_lev_ld = align_to_lead_day(t_lev)
        z_lev_ld = align_to_lead_day(z_lev)
        t_lev_ld, z_lev_ld, _ = align_common_lead_day([t_lev_ld, z_lev_ld, t2m_ld])
        if t_lev_ld is not None and z_lev_ld is not None:
            phi_lev = xr.where(z_lev_ld > 1e4, z_lev_ld, g * z_lev_ld)
            mse_star_levs.append((cp * t_lev_ld + Lv * _qsat(t_lev_ld, float(lev)) + phi_lev).assign_coords(level=lev))
    if not mse_star_levs:
        return None, None
    mse_star_all = xr.concat(mse_star_levs, dim="level")
    mse_star_masked = mse_star_all.where((mse_star_all["level"] > 300) & (mse_star_all["level"] <= lcl_p_grid))
    mse_star_max_grid = mse_star_masked.max(dim="level")
    barrier_grid = mse_star_max_grid - mse_s_grid
    spatial_dims_mse = [d for d in ["latitude", "longitude", "lat", "lon"] if d in mse_star_max_grid.dims]
    spatial_dims_bar = [d for d in ["latitude", "longitude", "lat", "lon"] if d in barrier_grid.dims]
    mse_star_max_mean = mse_star_max_grid.mean(dim=spatial_dims_mse)
    barrier_mean = barrier_grid.mean(dim=spatial_dims_bar)
    mse_val = np.asarray(mse_star_max_mean.mean(dim="lead_day").values).squeeze() / 1000.0
    bar_val = np.asarray(barrier_mean.mean(dim="lead_day").values).squeeze() / 1000.0
    if np.shape(mse_val) != (51,) or np.shape(bar_val) != (51,):
        return None, None
    return mse_val, bar_val

raw_records = []
daily_tmax_records = []
daily_member_raw_records = []

def append_daily_tmax_records(exp_start_date, da_tmax_daily, era5_tmax_daily):
    if da_tmax_daily is None:
        return
    target_dates = pd.date_range("2023-06-14", "2023-06-24", freq="D")
    for day in target_dates:
        date_str = day.strftime("%Y-%m-%d")
        if day not in pd.to_datetime(da_tmax_daily.time.values):
            continue
        s2s_day = da_tmax_daily.sel(time=day)
        era5_val = np.nan
        if era5_tmax_daily is not None and day in pd.to_datetime(era5_tmax_daily.time.values):
            era5_val = float(era5_tmax_daily.sel(time=day).values)
        vals = np.asarray(s2s_day.values).squeeze()
        if vals.shape != (51,):
            continue
        for member, value in enumerate(vals):
            daily_tmax_records.append({
                "init_date": exp_start_date,
                "date": date_str,
                "member": member,
                "s2s_tmax": value,
                "era5_tmax": era5_val,
            })

def get_daily_stage(day):
    day_ts = pd.Timestamp(day)
    for stage in ["Stage-I_dry", "Stage-II_wet"]:
        w_start, w_end = study_window_dict_run[stage]
        if pd.Timestamp(w_start) <= day_ts <= pd.Timestamp(w_end):
            return stage
    return None

def extract_daily_member_values(da_daily, day):
    if da_daily is None:
        return None
    if day not in pd.to_datetime(da_daily.time.values):
        return None
    vals = np.asarray(da_daily.sel(time=day).values).squeeze()
    return vals if vals.shape == (51,) else None

def append_daily_member_raw_factor_records(exp_start_date, init_factors, da_tmax_daily, da_tp_daily, da_sm_daily, da_ssr_daily, da_shf_daily, da_z500_daily, da_wnpsh_daily):
    target_dates = pd.date_range("2023-06-14", "2023-06-24", freq="D")
    for day in target_dates:
        stage = get_daily_stage(day)
        if stage is None:
            continue
        daily_arrays = {
            "y_tmax": extract_daily_member_values(da_tmax_daily, day),
            "NCHN_tp": extract_daily_member_values(da_tp_daily, day),
            "sm_avg": extract_daily_member_values(da_sm_daily, day),
            "SSR_Avg": extract_daily_member_values(da_ssr_daily, day),
            # SHF_Avg is stored as upward-positive; do not flip it again in Cell 2/3.
            "SHF_Avg": extract_daily_member_values(da_shf_daily, day),
            "z500_anom_NCHN": extract_daily_member_values(da_z500_daily, day),
            "WNPSH": extract_daily_member_values(da_wnpsh_daily, day),
        }
        shape_inputs = dict(daily_arrays)
        shape_inputs.update(init_factors)
        bad_shapes = {k: np.shape(v) for k, v in shape_inputs.items() if np.shape(v) != (51,)}
        if bad_shapes:
            print(f"  [Warning] daily member factor shape check failed for {exp_start_date} {day:%Y-%m-%d}: {bad_shapes}; skipped")
            continue
        for member in range(51):
            row = {"init_date": exp_start_date, "date": day.strftime("%Y-%m-%d"), "member": member, "stage": stage}
            for name, vals in daily_arrays.items():
                row[name] = vals[member]
            for name, vals in init_factors.items():
                row[name] = vals[member]
            daily_member_raw_records.append(row)

print("-> 正在显式预加载合并 CF+PF (51 Members) 核心数据阵列...")
da_tmax_all = load_s2s_ensemble_with_cf(f"{PF_DIR}/ecmf_rel_pf_sfc_mx2t6_2023-06.grb", f"{CF_DIR}/ecmf_cf_sfc_mx2t6_2023-06.grib", ["mx2t6", "mx2t", "tmax"])
da_tp_all = load_s2s_ensemble_with_cf(f"{PF_DIR}/ecmf_rel_pf_sfc_tp2023-06.grb", f"{CF_DIR}/ecmf_cf_sfc_tp_2023-06.grib", ["tp", "total_precipitation"])
da_sm_all = load_s2s_ensemble_with_cf(f"{PF_DIR}/ecmf_rel_pf_sfc_sm2023-06.grb", f"{CF_DIR}/ecmf_cf_sfc_sm20_2023-06.grib", ["sm20", "sm", "swvl1", "swvl2", "vwc1"])
da_ssr_all = load_s2s_ensemble_with_cf(f"{PF_DIR}/ecmf_rel_pf_sfc_radiation2023-06.grb", f"{CF_DIR}/ecmf_cf_sfc_sshf+slhf+snsr_2023-06.grib", ["snsr", "ssr", "ssrd", "surface_net_solar_radiation"])
da_sshf_all = load_s2s_ensemble_with_cf(f"{PF_DIR}/ecmf_rel_pf_sfc_hflux2023-06.grb", f"{CF_DIR}/ecmf_cf_sfc_sshf+slhf+snsr_2023-06.grib", ["sshf", "ishf", "surface_sensible_heat_flux"])
da_z500_all = load_s2s_ensemble_with_cf(f"{PF_DIR}/ecmf_rel_pf_pl_zqt2023-06.grb", f"{CF_DIR}/ecmf_cf_pl500_156_2023-06.grib", ["gh", "z", "geopotential"], level=500)
da_wnpsh_all = load_s2s_ensemble_with_cf(f"{PF_DIR}/ecmf_rel_pf_pl_zqt2023-06.grb", f"{CF_DIR}/ecmf_cf_pl850_156_2023-06.grib", ["gh", "z", "geopotential"], level=850)
era5_tmax_daily = load_era5_tmax_daily(ERA5_TMAX_FILE)

for exp_start_date in start_date_list_run:
    print(f"\n[Processing Start Date: {exp_start_date}]")
    init_factors = {}
    try:
        max_val, bar_val = compute_real_mse(exp_start_date)
        ncvi_val = compute_real_ncvi(exp_start_date)
        init_factors["Initial_MSEstar_max_NCHN"] = max_val
        init_factors["Initial_Barrier_NCHN"] = bar_val
        init_factors["NCVI"] = ncvi_val
    except Exception as e:
        print(f"  [Warning] Initial 因子推算异常: {e}")
    bad_shapes = {k: np.shape(v) for k, v in init_factors.items() if np.shape(v) != (51,)}
    if bad_shapes:
        print(f"  [Warning] 初始因子 shape 不符合要求 (51,)，跳过该起报日: {bad_shapes}")
        continue

    tmp_tmax = get_s2s_init_slice(da_tmax_all.to_dataset(name="v"), "v", exp_start_date)
    da_tmax_m = process_s2s_tmax_bjt_daily_max(tmp_tmax, region=NCHN_BOX)
    append_daily_tmax_records(exp_start_date, da_tmax_m, era5_tmax_daily)
    tmp_tp = get_s2s_init_slice(da_tp_all.to_dataset(name="v"), "v", exp_start_date)
    da_tp_m = process_accum_variable(tmp_tp, "precip", region=NCHN_BOX)
    tmp_sm = get_s2s_init_slice(da_sm_all.to_dataset(name="v"), "v", exp_start_date)
    da_sm_m = process_s2s_bjt_daily_mean(tmp_sm, "soil_moisture", region=NCHN_BOX)
    tmp_ssr = get_s2s_init_slice(da_ssr_all.to_dataset(name="v"), "v", exp_start_date)
    da_ssr_m = process_accum_variable(tmp_ssr, "flux_ssr", region=NCHN_BOX)
    tmp_sshf = get_s2s_init_slice(da_sshf_all.to_dataset(name="v"), "v", exp_start_date)
    da_sshf_m = process_accum_variable(tmp_sshf, "flux_shf", region=NCHN_BOX)
    tmp_z500 = get_s2s_init_slice(da_z500_all.to_dataset(name="v"), "v", exp_start_date)
    da_z500_raw_m = process_s2s_bjt_daily_mean(tmp_z500, "z500", region=NCHN_BOX)
    try:
        z500_clim = load_z500_reforecast_climatology(exp_start_date, region=NCHN_BOX)
        da_z500_m = apply_z500_anomaly(da_z500_raw_m, z500_clim)
    except Exception:
        da_z500_m = None
    tmp_wnpsh = get_s2s_init_slice(da_wnpsh_all.to_dataset(name="v"), "v", exp_start_date)
    da_wnpsh_m = process_s2s_bjt_daily_mean(tmp_wnpsh, "wnpsh", region=WNPSH_BOX)

    append_daily_member_raw_factor_records(exp_start_date, init_factors, da_tmax_m, da_tp_m, da_sm_m, da_ssr_m, da_sshf_m, da_z500_m, da_wnpsh_m)

    for stage, (win_start, win_end) in study_window_dict_run.items():
        try:
            y_tmax = extract_window_only(da_tmax_m, win_start, win_end, agg_func="mean")
            dyn_factors = {
                "NCHN_tp": extract_window_only(da_tp_m, win_start, win_end, agg_func="sum"),
                "sm_avg": extract_window_only(da_sm_m, win_start, win_end, agg_func="mean"),
                "SSR_Avg": extract_window_only(da_ssr_m, win_start, win_end, agg_func="mean"),
                "SHF_Avg": extract_window_only(da_sshf_m, win_start, win_end, agg_func="mean"),
                "z500_anom_NCHN": extract_window_only(da_z500_m, win_start, win_end, agg_func="mean"),
                "WNPSH": extract_window_only(da_wnpsh_m, win_start, win_end, agg_func="mean"),
            }
        except Exception:
            continue
        shapes_to_check = {"y_tmax": y_tmax}
        shapes_to_check.update(dyn_factors)
        shapes_to_check.update(init_factors)
        bad_shapes = {k: np.shape(v) for k, v in shapes_to_check.items() if np.shape(v) != (51,)}
        if bad_shapes:
            continue
        for i in range(51):
            row = {"init_date": exp_start_date, "stage": stage, "member": i, "y_tmax": y_tmax[i]}
            for f in factor_cols_raw:
                if f in init_factors:
                    row[f] = init_factors[f][i]
                elif f in dyn_factors:
                    row[f] = dyn_factors[f][i]
            raw_records.append(row)

df_raw_all = pd.DataFrame(raw_records).dropna().reset_index(drop=True)
raw_csv = f"{SUB_DIRS['tables']}/table_stage_member_raw_factors_SHF.csv"
df_raw_all.to_csv(raw_csv, index=False)
# SHF_Avg is stored as upward-positive in the raw factor table.
df_raw_all.groupby(["init_date", "stage"])["SHF_Avg"].agg(["count", "mean", "std", "min", "max"]).reset_index().to_csv(
    f"{SUB_DIRS['tables']}/table_stage_member_raw_factors_SHF_check.csv", index=False
)
pd.DataFrame(daily_tmax_records).to_csv(f"{SUB_DIRS['tables']}/table_daily_tmax_SHF.csv", index=False)
pd.DataFrame(daily_member_raw_records).to_csv(f"{SUB_DIRS['tables']}/table_daily_member_raw_factors_SHF.csv", index=False)
if len(df_raw_all) != EXPECTED_RAW_ROWS:
    raise RuntimeError(f"raw table row check failed: expected {EXPECTED_RAW_ROWS}, got {len(df_raw_all)}; file={raw_csv}")
print(f"Cell 1 complete: wrote {raw_csv} ({len(df_raw_all)} rows).")



In [ ]:
# =============================================================================
# Cell 2: Read raw CSV; run stage-specific residualization + OLS + LMG; write tables
# =============================================================================
import ast
import os
import warnings
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import pearsonr
from sklearn.linear_model import LinearRegression

warnings.filterwarnings("ignore", category=FutureWarning)

print("\n" + "=" * 80)
print("Cell 2: residualization + OLS + LMG from raw CSV only...")
print("=" * 80)

EXP_OUT_DIR = "/data1/huangy/fig6/NC/v6"
RESID_OLS_DIR = f"{EXP_OUT_DIR}/stage_specific_residualized_ols_SHF_0608_0612_v2"
SUB_DIRS = {
    "tables": f"{RESID_OLS_DIR}/tables",
    "corr_raw": f"{RESID_OLS_DIR}/figures/corr_raw",
    "corr_resid": f"{RESID_OLS_DIR}/figures/corr_resid",
    "lmg_main": f"{RESID_OLS_DIR}/figures/lmg_main",
    "lmg_sensitivity": f"{RESID_OLS_DIR}/figures/lmg_sensitivity",
    "ols_lmg_combo": f"{RESID_OLS_DIR}/figures/ols_lmg_combo",
    "tmax_timeseries": f"{RESID_OLS_DIR}/figures/tmax_timeseries",
    "metrics_heatmap": f"{RESID_OLS_DIR}/figures/metrics_heatmap",
}
for d in SUB_DIRS.values():
    os.makedirs(d, exist_ok=True)

start_date_list_run = ["2023-06-08", "2023-06-12"]
study_window_dict = {
    "Stage-I_dry": ("2023-06-14", "2023-06-17"),
    "Stage-II_wet": ("2023-06-18", "2023-06-24"),
    "Total": ("2023-06-14", "2023-06-24"),
}
study_window_dict_run = study_window_dict
resid_strategies = {
    "Stage-I_dry": "SM-led residualization",
    "Stage-II_wet": "TP-led residualization",
    # Mechanistic attribution focuses on Stage-I and Stage-II; Total is retained only as an overall diagnostic.
    "Total": "TP-led residualization (Overall Diagnostic)",
}
EXPECTED_RAW_ROWS = len(start_date_list_run) * len(study_window_dict_run) * 51
EXPECTED_OLS_COMBINATIONS = len(start_date_list_run) * len(study_window_dict_run)
RAW_MODEL_FACTORS = [
    "WNPSH", "z500_anom_NCHN", "NCHN_tp", "sm_avg", "SSR_Avg", "SHF_Avg",
    "Initial_MSEstar_max_NCHN", "Initial_Barrier_NCHN", "NCVI",
]

def calc_resid(y_col, x_cols, df):
    y_std = df[y_col].std()
    x_std_sum = df[x_cols].std().sum()
    if y_std == 0:
        return np.zeros(len(df))
    if x_std_sum == 0:
        return df[y_col] - df[y_col].mean()
    X = sm.add_constant(df[x_cols])
    return sm.OLS(df[y_col], X).fit().resid

def compute_lmg_mc(X, y, feature_names, mc_samples=2000):
    np.random.seed(42)
    n_samples, n_features = X.shape
    if n_samples < n_features + 2:
        return {name: np.nan for name in feature_names}
    X_s = (X - np.nanmean(X, axis=0)) / (np.nanstd(X, axis=0) + 1e-8)
    y_s = (y - np.nanmean(y)) / (np.nanstd(y) + 1e-8)
    lmg_raw = np.zeros(n_features)
    model = LinearRegression()
    for _ in range(mc_samples):
        order = np.random.permutation(n_features)
        prev_r2 = 0.0
        for i in range(n_features):
            idx = order[:i + 1]
            r2 = model.fit(X_s[:, idx], y_s).score(X_s[:, idx], y_s)
            lmg_raw[order[i]] += max(0, (r2 - prev_r2))
            prev_r2 = r2
    tot = lmg_raw.sum()
    if tot == 0:
        return {name: 0.0 for name in feature_names}
    return {name: val for name, val in zip(feature_names, (lmg_raw / tot) * 100.0)}

def _stringify_list(values):
    return ";".join(values) if values else ""

def safe_corr(a, b):
    if len(a) < 2 or np.nanstd(a) == 0 or np.nanstd(b) == 0:
        return np.nan
    return pearsonr(a, b)[0]

def fit_ols_lmg(df, factor_names, model_type, residualization_strategy):
    valid_factors = [f for f in factor_names if f in df.columns and df[f].std() >= 1e-12]
    dropped_factors = [f for f in factor_names if f not in valid_factors]
    X = df[valid_factors].values
    y = df["y_tmax"].values
    model = sm.OLS(y, sm.add_constant(X)).fit()
    y_pred = model.fittedvalues
    lmg_dict = compute_lmg_mc(X, y, valid_factors)
    sorted_lmg = sorted(lmg_dict.items(), key=lambda item: item[1], reverse=True)
    summary = {
        "init_date": df["init_date"].iloc[0],
        "stage": df["stage"].iloc[0],
        "model_type": model_type,
        "residualization_strategy": residualization_strategy,
        "R2": model.rsquared,
        "Adj_R2": model.rsquared_adj,
        "Pred_vs_ECMWF_Corr": safe_corr(y_pred, y),
        "Pred_vs_ECMWF_RMSE": np.sqrt(np.mean((y_pred - y) ** 2)),
        "Pred_vs_ECMWF_Bias": np.mean(y_pred - y),
        "Top1_factor_name": sorted_lmg[0][0] if sorted_lmg else np.nan,
        "Top1_factor_importance_pct": sorted_lmg[0][1] if sorted_lmg else np.nan,
        "dropped_factors": dropped_factors,
        "lmg_dict": lmg_dict,
    }
    fitted = df[["init_date", "stage", "member", "y_tmax"]].copy()
    fitted["model_type"] = model_type
    fitted["fitted_value"] = y_pred
    fitted["residual"] = y - y_pred
    return summary, lmg_dict, fitted

def build_factor_direction_rows(df, factor_names, model_kind):
    n_members = len(df)
    if n_members != 51:
        print(f"  [Warning] factor direction diagnostics for {df['init_date'].iloc[0]} | {df['stage'].iloc[0]} | {model_kind}: n_members={n_members}, expected 51")
    valid_factors = [f for f in factor_names if f in df.columns and df[f].std() >= 1e-12]
    if not valid_factors:
        return []
    y = df["y_tmax"].astype(float).values
    y_std = np.nanstd(y)
    y_s = (y - np.nanmean(y)) / (y_std + 1e-8)
    X = df[valid_factors].astype(float).values
    X_s = (X - np.nanmean(X, axis=0)) / (np.nanstd(X, axis=0) + 1e-8)
    model = sm.OLS(y_s, sm.add_constant(X_s)).fit()
    coef_map = {factor: model.params[i + 1] for i, factor in enumerate(valid_factors)}
    rows = []
    for factor in valid_factors:
        coef = float(coef_map[factor])
        rows.append({
            "init_date": df["init_date"].iloc[0],
            "stage": df["stage"].iloc[0],
            "model_kind": model_kind,
            "factor": factor,
            "pearson_r_with_y": safe_corr(df[factor].astype(float).values, y),
            "ols_coef_standardized": coef,
            "coef_sign": "positive" if coef > 0 else ("negative" if coef < 0 else "zero"),
            "n_members": n_members,
        })
    return rows

def build_lmg_long_rows(summary_row, lmg_dict):
    dropped = summary_row.get("dropped_factors", [])
    dropped_str = dropped if isinstance(dropped, str) else _stringify_list(dropped)
    return [
        {
            "init_date": summary_row["init_date"],
            "stage": summary_row["stage"],
            "model_type": summary_row["model_type"],
            "residualization_strategy": summary_row["residualization_strategy"],
            "factor_name": factor_name,
            "relative_importance_pct": importance,
            "R2": summary_row["R2"],
            "Adj_R2": summary_row["Adj_R2"],
            "dropped_factors": dropped_str,
        }
        for factor_name, importance in lmg_dict.items()
    ]

def stringify_summary_df(df):
    df_csv = df.copy()
    if not df_csv.empty:
        df_csv["dropped_factors"] = df_csv["dropped_factors"].apply(_stringify_list)
        df_csv["lmg_dict"] = df_csv["lmg_dict"].apply(repr)
    return df_csv

raw_csv = f"{SUB_DIRS['tables']}/table_stage_member_raw_factors_SHF.csv"
df_raw_all = pd.read_csv(raw_csv)
if len(df_raw_all) != EXPECTED_RAW_ROWS:
    raise RuntimeError(f"raw table row check failed: expected {EXPECTED_RAW_ROWS}, got {len(df_raw_all)}; file={raw_csv}")

resid_records = []
tp_param_resid_records = []
fitted_records = []
raw_ols_results = []
main_ols_results = []
sens_ols_results = []
raw_lmg_long_records = []
main_lmg_long_records = []
sens_lmg_long_records = []
factor_direction_records = []

for (exp_start_date, stage), df_raw in df_raw_all.groupby(["init_date", "stage"], sort=False):
    df_raw = df_raw.dropna().copy()
    if len(df_raw) != 51:
        raise RuntimeError(f"raw group size check failed for {exp_start_date}/{stage}: expected 51, got {len(df_raw)}")

    raw_summary, raw_lmg, raw_fitted = fit_ols_lmg(df_raw, RAW_MODEL_FACTORS, "raw", "No residualization")
    raw_ols_results.append(raw_summary)
    raw_lmg_long_records.extend(build_lmg_long_rows(raw_summary, raw_lmg))
    fitted_records.extend(raw_fitted.to_dict("records"))
    factor_direction_records.extend(build_factor_direction_rows(df_raw, RAW_MODEL_FACTORS, "raw"))

    df_resid = df_raw.copy()
    if stage == "Stage-I_dry":
        df_resid["SSR_resid"] = calc_resid("SSR_Avg", ["sm_avg"], df_resid)
        df_resid["SHF_resid"] = calc_resid("SHF_Avg", ["sm_avg"], df_resid)
        target_factors = ["WNPSH", "z500_anom_NCHN", "NCHN_tp", "sm_avg", "SSR_resid", "SHF_resid", "Initial_MSEstar_max_NCHN", "Initial_Barrier_NCHN", "NCVI"]
    else:
        # Stage-II uses TP-led residualization; Total uses the same TP-led form only as an overall diagnostic.
        df_resid["SM_resid"] = calc_resid("sm_avg", ["NCHN_tp"], df_resid)
        df_resid["SSR_resid"] = calc_resid("SSR_Avg", ["NCHN_tp"], df_resid)
        df_resid["SHF_resid"] = calc_resid("SHF_Avg", ["NCHN_tp"], df_resid)
        target_factors = ["WNPSH", "z500_anom_NCHN", "NCHN_tp", "SM_resid", "SSR_resid", "SHF_resid", "Initial_MSEstar_max_NCHN", "Initial_Barrier_NCHN", "NCVI"]

    resid_records.extend(df_resid.to_dict("records"))
    main_summary, main_lmg, main_fitted = fit_ols_lmg(df_resid, target_factors, "residualized_main", resid_strategies.get(stage, ""))
    main_ols_results.append(main_summary)
    main_lmg_long_records.extend(build_lmg_long_rows(main_summary, main_lmg))
    fitted_records.extend(main_fitted.to_dict("records"))
    factor_direction_records.extend(build_factor_direction_rows(df_resid, target_factors, "residualized_main"))

    if stage == "Stage-II_wet":
        df_sens = df_raw.copy()
        df_sens["TP_param_resid"] = calc_resid("NCHN_tp", ["z500_anom_NCHN", "WNPSH"], df_sens)
        df_sens["SM_param_resid"] = calc_resid("sm_avg", ["TP_param_resid"], df_sens)
        df_sens["SSR_param_resid"] = calc_resid("SSR_Avg", ["TP_param_resid"], df_sens)
        df_sens["SHF_param_resid"] = calc_resid("SHF_Avg", ["TP_param_resid"], df_sens)
        sens_factors = ["WNPSH", "z500_anom_NCHN", "TP_param_resid", "SM_param_resid", "SSR_param_resid", "SHF_param_resid", "Initial_MSEstar_max_NCHN", "Initial_Barrier_NCHN", "NCVI"]
        tp_param_resid_records.extend(df_sens.to_dict("records"))
        sens_summary, sens_lmg, sens_fitted = fit_ols_lmg(df_sens, sens_factors, "sensitivity_TPparam", "TP_param_resid-led sensitivity")
        sens_ols_results.append(sens_summary)
        sens_lmg_long_records.extend(build_lmg_long_rows(sens_summary, sens_lmg))
        fitted_records.extend(sens_fitted.to_dict("records"))
        factor_direction_records.extend(build_factor_direction_rows(df_sens, sens_factors, "sensitivity_TPparam"))

df_resid_all = pd.DataFrame(resid_records)
df_tp_param_resid_all = pd.DataFrame(tp_param_resid_records)
df_fitted_all = pd.DataFrame(fitted_records)
df_raw_ols = pd.DataFrame(raw_ols_results)
df_main_ols = pd.DataFrame(main_ols_results)
df_sens_ols = pd.DataFrame(sens_ols_results)
df_raw_lmg_long = pd.DataFrame(raw_lmg_long_records)
df_lmg_long = pd.DataFrame(main_lmg_long_records)
df_sens_lmg_long = pd.DataFrame(sens_lmg_long_records)
df_factor_direction = pd.DataFrame(factor_direction_records)

actual_n = len(df_main_ols)
print(f"\n[RESULT CHECK] successful combinations = {actual_n}/{EXPECTED_OLS_COMBINATIONS}")
if actual_n != EXPECTED_OLS_COMBINATIONS:
    raise RuntimeError(f"OLS combination check failed: expected {EXPECTED_OLS_COMBINATIONS}/{EXPECTED_OLS_COMBINATIONS}, got {actual_n}/{EXPECTED_OLS_COMBINATIONS}")

df_resid_all.to_csv(f"{SUB_DIRS['tables']}/table_stage_member_residualized_factors_SHF.csv", index=False)
df_fitted_all.to_csv(f"{SUB_DIRS['tables']}/table_stage_member_ols_fitted_SHF.csv", index=False)
stringify_summary_df(df_raw_ols).to_csv(f"{SUB_DIRS['tables']}/table_stage_raw_ols_lmg_SHF.csv", index=False)
df_raw_lmg_long.to_csv(f"{SUB_DIRS['tables']}/table_stage_raw_lmg_long_SHF.csv", index=False)
df_factor_direction.to_csv(f"{SUB_DIRS['tables']}/table_factor_direction_diagnostics_SHF.csv", index=False)
stringify_summary_df(df_main_ols).to_csv(f"{SUB_DIRS['tables']}/table_stage_ols_lmg_SHF.csv", index=False)
df_lmg_long.to_csv(f"{SUB_DIRS['tables']}/table_stage_lmg_long_SHF.csv", index=False)
if not df_sens_ols.empty:
    df_tp_param_resid_all.to_csv(f"{SUB_DIRS['tables']}/table_stage_member_TPparam_residualized_factors_SHF.csv", index=False)
    stringify_summary_df(df_sens_ols).to_csv(f"{SUB_DIRS['tables']}/table_stageII_TPparam_sensitivity_SHF.csv", index=False)
    df_sens_lmg_long.to_csv(f"{SUB_DIRS['tables']}/table_stageII_TPparam_sensitivity_lmg_long_SHF.csv", index=False)
print("Cell 2 complete: wrote raw/main/sensitivity OLS, fitted values, and long-format LMG tables.")



In [ ]:
# =============================================================================
# Cell 3: Read CSVs and draw correlation matrices, scatter+LMG, Tmax series, heatmaps
# =============================================================================
import ast
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as patches
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)

print("\n" + "=" * 80)
print("Cell 3: plot matrices, scatter+LMG composites, Tmax series, and heatmaps from CSV files only...")
print("=" * 80)

EXP_OUT_DIR = "/data1/huangy/fig6/NC/v6"
RESID_OLS_DIR = f"{EXP_OUT_DIR}/stage_specific_residualized_ols_SHF_0608_0612_v2"
ERA5_TMAX_FILE = "/data1/huangy/fig6/NC/ai_test/pangu/era5_T/era5_2m_temperature_6_2023.nc"
SUB_DIRS = {
    "tables": f"{RESID_OLS_DIR}/tables",
    "corr_raw": f"{RESID_OLS_DIR}/figures/corr_raw",
    "corr_resid": f"{RESID_OLS_DIR}/figures/corr_resid",
    "lmg_main": f"{RESID_OLS_DIR}/figures/lmg_main",
    "lmg_sensitivity": f"{RESID_OLS_DIR}/figures/lmg_sensitivity",
    "ols_lmg_combo": f"{RESID_OLS_DIR}/figures/ols_lmg_combo",
    "tmax_timeseries": f"{RESID_OLS_DIR}/figures/tmax_timeseries",
    "metrics_heatmap": f"{RESID_OLS_DIR}/figures/metrics_heatmap",
}
for d in SUB_DIRS.values():
    os.makedirs(d, exist_ok=True)

start_date_list_run = ["2023-06-08", "2023-06-12"]
study_window_dict = {
    "Stage-I_dry": ("2023-06-14", "2023-06-17"),
    "Stage-II_wet": ("2023-06-18", "2023-06-24"),
    "Total": ("2023-06-14", "2023-06-24"),
}
study_window_dict_run = study_window_dict
resid_strategies = {
    "Stage-I_dry": "SM-led residualization",
    "Stage-II_wet": "TP-led residualization",
    "Total": "TP-led residualization (Overall Diagnostic)",
}
EXPECTED_RAW_ROWS = len(start_date_list_run) * len(study_window_dict_run) * 51
EXPECTED_OLS_COMBINATIONS = len(start_date_list_run) * len(study_window_dict_run)
DAILY_PROJECTION_STAGES = ["Stage-I_dry", "Stage-II_wet"]
EXPECTED_DAILY_PROJECTION_ROWS = len(start_date_list_run) * (4 + 7)
EXPECTED_DAILY_PROJECTION_STAGE_DAYS = {"Stage-I_dry": 4, "Stage-II_wet": 7}

raw_csv = f"{SUB_DIRS['tables']}/table_stage_member_raw_factors_SHF.csv"
resid_csv = f"{SUB_DIRS['tables']}/table_stage_member_residualized_factors_SHF.csv"
fitted_csv = f"{SUB_DIRS['tables']}/table_stage_member_ols_fitted_SHF.csv"
daily_csv = f"{SUB_DIRS['tables']}/table_daily_tmax_SHF.csv"
daily_member_raw_csv = f"{SUB_DIRS['tables']}/table_daily_member_raw_factors_SHF.csv"
daily_ols_projection_csv = f"{SUB_DIRS['tables']}/table_daily_tmax_with_ols_projection_SHF.csv"
daily_projection_consistency_csv = f"{SUB_DIRS['tables']}/table_daily_ols_projection_consistency_SHF.csv"
daily_projection_setup_csv = f"{SUB_DIRS['tables']}/table_daily_ols_projection_model_setup_SHF.csv"
daily_stage_predictor_compare_csv = f"{SUB_DIRS['tables']}/table_daily_to_stage_predictor_comparison_SHF.csv"
raw_ols_csv = f"{SUB_DIRS['tables']}/table_stage_raw_ols_lmg_SHF.csv"
main_ols_csv = f"{SUB_DIRS['tables']}/table_stage_ols_lmg_SHF.csv"
sens_ols_csv = f"{SUB_DIRS['tables']}/table_stageII_TPparam_sensitivity_SHF.csv"
factor_direction_csv = f"{SUB_DIRS['tables']}/table_factor_direction_diagnostics_SHF.csv"
raw_shf_check_csv = f"{SUB_DIRS['tables']}/table_stage_member_raw_factors_SHF_check.csv"

for stale_daily_projection_csv in [
    daily_ols_projection_csv,
    daily_projection_consistency_csv,
    daily_projection_setup_csv,
    daily_stage_predictor_compare_csv,
]:
    if os.path.exists(stale_daily_projection_csv):
        os.remove(stale_daily_projection_csv)
        print(f"[Cell 3] Removed stale daily projection output: {stale_daily_projection_csv}")
for init in start_date_list_run:
    for stale_tmax_fig in [
        f"{SUB_DIRS['tmax_timeseries']}/tmax_timeseries_{init}_SHF.png",
        f"{SUB_DIRS['tmax_timeseries']}/tmax_timeseries_stage_mean_diagnostic_{init}_SHF.png",
    ]:
        if os.path.exists(stale_tmax_fig):
            os.remove(stale_tmax_fig)
            print(f"[Cell 3] Removed stale Tmax time-series figure: {stale_tmax_fig}")

df_raw_all = pd.read_csv(raw_csv)
df_resid_all = pd.read_csv(resid_csv)
df_fitted_all = pd.read_csv(fitted_csv)
df_daily_tmax = pd.read_csv(daily_csv) if os.path.exists(daily_csv) else pd.DataFrame()
df_daily_member_raw = pd.read_csv(daily_member_raw_csv) if os.path.exists(daily_member_raw_csv) else pd.DataFrame()
if not df_daily_member_raw.empty:
    df_daily_member_raw["date"] = pd.to_datetime(df_daily_member_raw["date"])
df_raw_ols = pd.read_csv(raw_ols_csv)
df_main_ols = pd.read_csv(main_ols_csv)
df_sens_ols = pd.read_csv(sens_ols_csv) if os.path.exists(sens_ols_csv) else pd.DataFrame()
df_factor_direction = pd.read_csv(factor_direction_csv) if os.path.exists(factor_direction_csv) else pd.DataFrame()
df_raw_shf_check = pd.read_csv(raw_shf_check_csv) if os.path.exists(raw_shf_check_csv) else pd.DataFrame()
print(f"[Cell 3] ERA5 Tmax file path used for daily metrics: {ERA5_TMAX_FILE}")
if len(df_raw_all) != EXPECTED_RAW_ROWS:
    raise RuntimeError(f"raw table row check failed: expected {EXPECTED_RAW_ROWS}, got {len(df_raw_all)}; file={raw_csv}")
if len(df_main_ols) != EXPECTED_OLS_COMBINATIONS:
    raise RuntimeError(f"OLS combination check failed: expected {EXPECTED_OLS_COMBINATIONS}/{EXPECTED_OLS_COMBINATIONS}, got {len(df_main_ols)}/{EXPECTED_OLS_COMBINATIONS}")

print("\n[SHF_Avg(up) range check] SHF_Avg is upward-positive; Cell 2/3 do not flip the sign again.")
if not df_raw_shf_check.empty:
    for _, row in df_raw_shf_check.iterrows():
        print(f"  {row['init_date']} | {row['stage']}: SHF_Avg min={row['min']:.2f}, max={row['max']:.2f}, mean={row['mean']:.2f}, std={row['std']:.2f}")
        if row['mean'] < 0:
            print("    [Warning] SHF_Avg(up) mean is negative; verify upward-positive convention and input sign.")
        if abs(row['mean']) > 1000 or abs(row['min']) > 3000 or abs(row['max']) > 3000:
            print("    [Warning] SHF_Avg(up) range appears unusually large; please inspect source flux units.")
else:
    shf_stats = df_raw_all.groupby(["init_date", "stage"])["SHF_Avg"].agg(["min", "max", "mean", "std"]).reset_index()
    for _, row in shf_stats.iterrows():
        print(f"  {row['init_date']} | {row['stage']}: SHF_Avg min={row['min']:.2f}, max={row['max']:.2f}, mean={row['mean']:.2f}, std={row['std']:.2f}")

plt.rcParams.update({"font.family": "sans-serif", "font.size": 12, "axes.linewidth": 1.2})
label_map = {
    "y_tmax": "Tmax",
    "SHF_Avg": "SHF_Avg(up)", "SHF_resid": "SHF_resid", "SHF_param_resid": "SHF_param_resid",
    "SM_resid": "SM_resid", "SM_param_resid": "SM_param_resid", "SSR_resid": "SSR_resid",
    "SSR_param_resid": "SSR_param_resid", "NCHN_tp": "NCHN_tp", "z500_anom_NCHN": "z500_anom",
    "Initial_MSEstar_max_NCHN": "Initial_MSEstar_max", "Initial_Barrier_NCHN": "Initial_Barrier",
    "NCVI": "NCVI", "WNPSH": "WNPSH", "sm_avg": "sm_avg", "TP_param_resid": "TP_param_resid",
}

def parse_lmg_dict(value):
    if isinstance(value, dict):
        return value
    if pd.isna(value):
        return {}
    parsed = ast.literal_eval(str(value))
    return parsed if isinstance(parsed, dict) else {}

def safe_corr_r2(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    mask = np.isfinite(a) & np.isfinite(b)
    if mask.sum() < 2 or np.nanstd(a[mask]) == 0 or np.nanstd(b[mask]) == 0:
        return np.nan
    return float(np.corrcoef(a[mask], b[mask])[0, 1] ** 2)

def clean_stage_name(stage):
    return stage.replace("Stage-I_dry", "Stage-I").replace("Stage-II_wet", "Stage-II")

def stage_day_count(stage):
    w_start, w_end = study_window_dict_run[stage]
    return len(pd.date_range(w_start, w_end, freq="D"))

def _fit_linear_coefficients(y, X):
    y_arr = np.asarray(y, dtype=float)
    X_arr = np.asarray(X, dtype=float)
    if y_arr.ndim != 1:
        y_arr = y_arr.squeeze()
    if X_arr.ndim == 1:
        X_arr = X_arr.reshape(-1, 1)
    if np.nanstd(y_arr) == 0:
        return np.array([float(np.nanmean(y_arr))] + [0.0] * X_arr.shape[1])
    if np.nansum(np.nanstd(X_arr, axis=0)) == 0:
        return np.array([float(np.nanmean(y_arr))] + [0.0] * X_arr.shape[1])
    design = np.column_stack([np.ones(len(y_arr)), X_arr])
    return np.linalg.lstsq(design, y_arr, rcond=None)[0]

def _apply_linear_coefficients(beta, X):
    X_arr = np.asarray(X, dtype=float)
    if X_arr.ndim == 1:
        X_arr = X_arr.reshape(-1, 1)
    design = np.column_stack([np.ones(len(X_arr)), X_arr])
    return design @ beta

def residualized_main_factors_for_stage(stage):
    if stage == "Stage-I_dry":
        return ["WNPSH", "z500_anom_NCHN", "NCHN_tp", "sm_avg", "SSR_resid", "SHF_resid", "Initial_MSEstar_max_NCHN", "Initial_Barrier_NCHN", "NCVI"]
    return ["WNPSH", "z500_anom_NCHN", "NCHN_tp", "SM_resid", "SSR_resid", "SHF_resid", "Initial_MSEstar_max_NCHN", "Initial_Barrier_NCHN", "NCVI"]

def raw_needed_cols_for_projection(stage):
    if stage == "Stage-I_dry":
        return ["y_tmax", "SSR_Avg", "SHF_Avg", "sm_avg"]
    return ["y_tmax", "sm_avg", "SSR_Avg", "SHF_Avg", "NCHN_tp"]

def daily_needed_cols_for_projection(stage):
    return [
        "y_tmax", "NCHN_tp", "sm_avg", "SSR_Avg", "SHF_Avg",
        "z500_anom_NCHN", "WNPSH", "Initial_MSEstar_max_NCHN",
        "Initial_Barrier_NCHN", "NCVI",
    ]

def fit_stage_projection_components(init, stage):
    raw_stage = df_raw_all[(df_raw_all["init_date"] == init) & (df_raw_all["stage"] == stage)].copy()
    resid_stage = df_resid_all[(df_resid_all["init_date"] == init) & (df_resid_all["stage"] == stage)].copy()
    raw_before = len(raw_stage)
    resid_before = len(resid_stage)
    factors = residualized_main_factors_for_stage(stage)

    needed_raw_cols = raw_needed_cols_for_projection(stage)
    missing_raw_cols = [c for c in needed_raw_cols if c not in raw_stage.columns]
    if missing_raw_cols:
        print(f"  [Daily projection debug] {init} | {stage}: raw_stage rows before subset dropna={raw_before}, after subset dropna=0, missing_raw_cols={missing_raw_cols}")
        return None, f"missing raw columns: {missing_raw_cols}"
    raw_stage = raw_stage.dropna(subset=needed_raw_cols).copy()

    needed_resid_cols = ["y_tmax"] + factors
    needed_resid_cols = [c for c in needed_resid_cols if c in resid_stage.columns]
    resid_stage = resid_stage.dropna(subset=needed_resid_cols).copy()
    print(f"  [Daily projection debug] {init} | {stage}: raw_stage rows before subset dropna={raw_before}, after subset dropna={len(raw_stage)}; resid_stage rows before subset dropna={resid_before}, after subset dropna={len(resid_stage)}")

    valid_factors = [f for f in factors if f in resid_stage.columns and resid_stage[f].std() >= 1e-12]
    if raw_stage.empty:
        return None, f"raw_stage empty after subset dropna on {needed_raw_cols} (before={raw_before})"
    if resid_stage.empty:
        return None, f"resid_stage empty after subset dropna on {needed_resid_cols} (before={resid_before})"
    if not valid_factors:
        return None, f"valid_factors empty after subset dropna; candidate factors={factors}"
    ols_beta = _fit_linear_coefficients(resid_stage["y_tmax"].values, resid_stage[valid_factors].values)
    resid_models = {}
    if stage == "Stage-I_dry":
        resid_models["SSR_resid"] = ("SSR_Avg", ["sm_avg"], _fit_linear_coefficients(raw_stage["SSR_Avg"].values, raw_stage[["sm_avg"]].values))
        resid_models["SHF_resid"] = ("SHF_Avg", ["sm_avg"], _fit_linear_coefficients(raw_stage["SHF_Avg"].values, raw_stage[["sm_avg"]].values))
    else:
        resid_models["SM_resid"] = ("sm_avg", ["NCHN_tp"], _fit_linear_coefficients(raw_stage["sm_avg"].values, raw_stage[["NCHN_tp"]].values))
        resid_models["SSR_resid"] = ("SSR_Avg", ["NCHN_tp"], _fit_linear_coefficients(raw_stage["SSR_Avg"].values, raw_stage[["NCHN_tp"]].values))
        resid_models["SHF_resid"] = ("SHF_Avg", ["NCHN_tp"], _fit_linear_coefficients(raw_stage["SHF_Avg"].values, raw_stage[["NCHN_tp"]].values))
    return {
        "ols_beta": ols_beta,
        "valid_factors": valid_factors,
        "resid_models": resid_models,
        "stage_fitted_mean": float(_apply_linear_coefficients(ols_beta, resid_stage[valid_factors].values).mean()),
        "ols_intercept": float(ols_beta[0]),
    }, "ok"

def prepare_daily_predictors_for_stage(daily_group, stage, components):
    df_proj = daily_group.copy()
    # Stage OLS uses window-total precipitation. For daily plotting, scale daily precipitation by
    # the number of days so the stage-mean of daily predictors equals the stage-level predictor.
    if "NCHN_tp" in df_proj.columns:
        df_proj["NCHN_tp"] = df_proj["NCHN_tp"].astype(float) * stage_day_count(stage)
    for resid_name, (y_col, x_cols, beta) in components["resid_models"].items():
        fitted_common = _apply_linear_coefficients(beta, df_proj[x_cols].astype(float).values)
        df_proj[resid_name] = df_proj[y_col].astype(float).values - fitted_common
    return df_proj

def build_daily_ols_projection(init, daily_stats, stage_mean_rows):
    if df_daily_member_raw.empty:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
    projection_rows = []
    setup_rows = []
    stage_mean_lookup = {(row["stage"]): row for row in stage_mean_rows if row["init_date"] == init}
    components_by_stage = {}
    for stage in DAILY_PROJECTION_STAGES:
        components, reason = fit_stage_projection_components(init, stage)
        if components is None:
            message = f"Daily projection setup failed for {init} | {stage}: {reason}"
            print(f"  [Warning] {message}")
            if stage == "Stage-I_dry":
                raise RuntimeError(message)
            continue
        components_by_stage[stage] = components
        setup_rows.append({
                "init_date": init,
                "stage": stage,
                "same_ols_coefficients_as_stage_model": True,
                "same_intercept_as_stage_model": True,
                "same_scaler_as_stage_model": "No scaler used in stage OLS",
                "same_residualization_equations_as_stage_model": True,
                "same_factor_sign_convention": "SHF_Avg upward-positive; no Cell 3 sign flip",
                "ols_intercept": components["ols_intercept"],
                "valid_factors": ";".join(components["valid_factors"]),
                "residualization_equations": ";".join([f"{name}={y_col}~{'+'.join(x_cols)}" for name, (y_col, x_cols, _) in components["resid_models"].items()]),
                "baseline_double_added": False,
                "prediction_formula": "X_daily_stage_scaled @ stage_OLS_beta; no extra stage mean/baseline added",
            })
    for day in sorted(daily_stats["date"].unique()):
        day_ts = pd.Timestamp(day)
        daily_group = df_daily_member_raw[(df_daily_member_raw["init_date"] == init) & (df_daily_member_raw["date"] == day_ts)].copy()
        if daily_group.empty:
            expected_stage = None
            for candidate_stage in DAILY_PROJECTION_STAGES:
                w_start, w_end = study_window_dict_run[candidate_stage]
                if pd.Timestamp(w_start) <= day_ts <= pd.Timestamp(w_end):
                    expected_stage = candidate_stage
                    break
            message = f"daily_group empty for {init} | {day_ts:%Y-%m-%d} | expected_stage={expected_stage}"
            print(f"  [Warning] {message}")
            if expected_stage == "Stage-I_dry":
                raise RuntimeError(message)
            continue
        stage = daily_group["stage"].iloc[0]
        daily_needed_cols = daily_needed_cols_for_projection(stage)
        missing_daily_cols = [c for c in daily_needed_cols if c not in daily_group.columns]
        if missing_daily_cols:
            message = f"daily_group missing required columns for {init} | {day_ts:%Y-%m-%d} | stage={stage}: {missing_daily_cols}"
            print(f"  [Warning] {message}")
            if stage == "Stage-I_dry":
                raise RuntimeError(message)
            continue
        daily_before = len(daily_group)
        daily_group = daily_group.dropna(subset=daily_needed_cols).copy()
        if daily_group.empty:
            message = f"daily_group empty after subset dropna for {init} | {day_ts:%Y-%m-%d} | stage={stage}: before={daily_before}, subset={daily_needed_cols}"
            print(f"  [Warning] {message}")
            if stage == "Stage-I_dry":
                raise RuntimeError(message)
            continue
        components = components_by_stage.get(stage)
        if components is None:
            message = f"projection components missing for {init} | {day_ts:%Y-%m-%d} | stage={stage}"
            print(f"  [Warning] {message}")
            if stage == "Stage-I_dry":
                raise RuntimeError(message)
            continue
        daily_resid = prepare_daily_predictors_for_stage(daily_group, stage, components)
        fitted_daily = _apply_linear_coefficients(components["ols_beta"], daily_resid[components["valid_factors"]].astype(float).values)
        stats_row = daily_stats[daily_stats["date"] == day_ts]
        if stats_row.empty:
            message = f"daily_stats empty for {init} | {day_ts:%Y-%m-%d} | stage={stage}"
            print(f"  [Warning] {message}")
            if stage == "Stage-I_dry":
                raise RuntimeError(message)
            continue
        stage_mean = stage_mean_lookup.get(stage, {})
        projection_rows.append({
            "init_date": init,
            "date": day_ts,
            "stage": stage,
            "ERA5_absT": float(stats_row.iloc[0]["ERA5_absT"]),
            "S2S_mean_absT": float(stats_row.iloc[0]["S2S_mean_absT"]),
            "S2S_spread": float(stats_row.iloc[0]["Spread_S2S"]),
            "OLS_daily_projection": float(np.nanmean(fitted_daily)),
            "OLS_daily_member_spread": float(np.nanstd(fitted_daily, ddof=1)),
            "OLS_stage_mean_absT": float(stage_mean.get("OLS_stage_mean_absT", np.nan)),
        })
    # Debug: reaggregate daily predictors back to stage scale and compare with Cell 2 stage predictors.
    comparison_rows = []
    for stage, components in components_by_stage.items():
        raw_stage = df_raw_all[(df_raw_all["init_date"] == init) & (df_raw_all["stage"] == stage)].copy()
        daily_stage = df_daily_member_raw[(df_daily_member_raw["init_date"] == init) & (df_daily_member_raw["stage"] == stage)].copy()
        if raw_stage.empty or daily_stage.empty:
            continue
        agg_map = {"NCHN_tp": "sum"}
        compare_cols = ["y_tmax", "sm_avg", "SSR_Avg", "SHF_Avg", "z500_anom_NCHN", "WNPSH", "Initial_MSEstar_max_NCHN", "Initial_Barrier_NCHN", "NCVI"]
        for col in compare_cols:
            if col in daily_stage.columns:
                agg_map[col] = "mean"
        daily_agg = daily_stage.groupby("member").agg(agg_map).reset_index()
        merged_raw = raw_stage[["member"] + [c for c in agg_map if c in raw_stage.columns]].merge(daily_agg, on="member", suffixes=("_stage", "_daily_reagg"))
        for col in agg_map:
            diff = merged_raw[f"{col}_daily_reagg"] - merged_raw[f"{col}_stage"]
            comparison_rows.append({"init_date": init, "stage": stage, "predictor_space": "raw", "factor": col, "max_abs_diff": float(np.nanmax(np.abs(diff))), "mean_abs_diff": float(np.nanmean(np.abs(diff)))})
        daily_resid_parts = []
        for _, day_group in daily_stage.groupby("date"):
            daily_resid_parts.append(prepare_daily_predictors_for_stage(day_group, stage, components))
        daily_resid_all = pd.concat(daily_resid_parts, ignore_index=True) if daily_resid_parts else pd.DataFrame()
        if not daily_resid_all.empty:
            resid_agg_map = {"NCHN_tp": "mean"}
            for col in [c for c in components["valid_factors"] if c != "NCHN_tp"]:
                if col in daily_resid_all.columns:
                    resid_agg_map[col] = "mean"
            daily_resid_agg = daily_resid_all.groupby("member").agg(resid_agg_map).reset_index()
            resid_stage = df_resid_all[(df_resid_all["init_date"] == init) & (df_resid_all["stage"] == stage)].copy()
            merged_resid = resid_stage[["member"] + [c for c in resid_agg_map if c in resid_stage.columns]].merge(daily_resid_agg, on="member", suffixes=("_stage", "_daily_reagg"))
            for col in resid_agg_map:
                diff = merged_resid[f"{col}_daily_reagg"] - merged_resid[f"{col}_stage"]
                comparison_rows.append({"init_date": init, "stage": stage, "predictor_space": "residualized_main", "factor": col, "max_abs_diff": float(np.nanmax(np.abs(diff))), "mean_abs_diff": float(np.nanmean(np.abs(diff)))})
    return pd.DataFrame(projection_rows), pd.DataFrame(setup_rows), pd.DataFrame(comparison_rows)

def display_corr_matrix(df, cols):
    corr = df[cols].corr()
    display_names = {col: label_map.get(col, col) for col in cols}
    return corr.rename(index=display_names, columns=display_names)

def truncate_colormap(cmap_name, minval=0.0, maxval=1.0, n=256):
    base = plt.get_cmap(cmap_name)
    return mcolors.LinearSegmentedColormap.from_list(
        f"{cmap_name}_trunc_{minval:.2f}_{maxval:.2f}",
        base(np.linspace(minval, maxval, n)),
    )

def lighten_color(color, amount=0.6):
    # amount 越大越接近白色；只改变绘图颜色，不改变任何表格数值。
    c = np.array(mcolors.to_rgb(color))
    return tuple(c + (1.0 - c) * amount)

def readable_text_color(facecolor):
    rgba = mcolors.to_rgba(facecolor)
    r, g, b = rgba[:3]
    luminance = 0.2126 * r + 0.7152 * g + 0.0722 * b
    return "white" if luminance < 0.35 else "black"

# 1. Correlation matrices from raw/residualized CSVs.
for (init, stg), group in df_raw_all.groupby(["init_date", "stage"]):
    fig, ax = plt.subplots(figsize=(10, 8))
    raw_cols = ["y_tmax", "NCHN_tp", "sm_avg", "SSR_Avg", "SHF_Avg", "z500_anom_NCHN", "WNPSH", "Initial_MSEstar_max_NCHN", "Initial_Barrier_NCHN", "NCVI"]
    raw_cols = [c for c in raw_cols if c in group.columns]
    sns.heatmap(display_corr_matrix(group, raw_cols), annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, ax=ax, square=True)
    ax.set_title(f"Raw Member-axis Correlation | Init {init} | {stg}", fontsize=14)
    fig.savefig(f"{SUB_DIRS['corr_raw']}/corr_raw_{init}_{stg}_SHF.png", dpi=200, bbox_inches="tight")
    plt.close(fig)

    resid_group = df_resid_all[(df_resid_all["init_date"] == init) & (df_resid_all["stage"] == stg)]
    if not resid_group.empty:
        if stg == "Stage-I_dry":
            res_cols = ["y_tmax", "NCHN_tp", "sm_avg", "SSR_resid", "SHF_resid", "z500_anom_NCHN", "WNPSH", "Initial_MSEstar_max_NCHN", "Initial_Barrier_NCHN", "NCVI"]
        else:
            res_cols = ["y_tmax", "NCHN_tp", "SM_resid", "SSR_resid", "SHF_resid", "z500_anom_NCHN", "WNPSH", "Initial_MSEstar_max_NCHN", "Initial_Barrier_NCHN", "NCVI"]
        res_cols = [c for c in res_cols if c in resid_group.columns]
        fig, ax = plt.subplots(figsize=(10, 8))
        sns.heatmap(display_corr_matrix(resid_group, res_cols), annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, ax=ax, square=True)
        ax.set_title(f"Residualized Correlation | Init {init} | {stg} | {resid_strategies.get(stg, '')}", fontsize=14)
        fig.savefig(f"{SUB_DIRS['corr_resid']}/corr_resid_{init}_{stg}_SHF.png", dpi=200, bbox_inches="tight")
        plt.close(fig)

# 2. Combined OLS scatter + LMG bar figures.
def plot_ols_lmg_combo(summary_row, color, filename):
    init, stg, model_type = summary_row["init_date"], summary_row["stage"], summary_row["model_type"]
    lmg_dict = parse_lmg_dict(summary_row["lmg_dict"])
    fit_group = df_fitted_all[(df_fitted_all["init_date"] == init) & (df_fitted_all["stage"] == stg) & (df_fitted_all["model_type"] == model_type)]
    if fit_group.empty or not lmg_dict:
        return
    fig, axes = plt.subplots(1, 2, figsize=(12, 5), gridspec_kw={"width_ratios": [1.05, 1.25]})
    axes[0].scatter(fit_group["y_tmax"], fit_group["fitted_value"], c=color, edgecolor="black", alpha=0.82)
    lim_min = float(np.nanmin([fit_group["y_tmax"].min(), fit_group["fitted_value"].min()]))
    lim_max = float(np.nanmax([fit_group["y_tmax"].max(), fit_group["fitted_value"].max()]))
    pad = max((lim_max - lim_min) * 0.08, 0.5)
    axes[0].plot([lim_min - pad, lim_max + pad], [lim_min - pad, lim_max + pad], "k--", lw=1)
    axes[0].set_xlim(lim_min - pad, lim_max + pad)
    axes[0].set_ylim(lim_min - pad, lim_max + pad)
    axes[0].set_xlabel("ECMWF member Tmax (°C)")
    axes[0].set_ylabel("OLS fitted Tmax (°C)")
    axes[0].set_title(f"Predicted vs ECMWF | R²={summary_row['R2']:.2f}, RMSE={summary_row['Pred_vs_ECMWF_RMSE']:.2f}")

    facs, imps = zip(*sorted(lmg_dict.items(), key=lambda item: item[1]))
    clean_facs = [label_map.get(f, f) for f in facs]
    bars = axes[1].barh(clean_facs, imps, color=color, edgecolor="black")
    axes[1].set_xlabel("LMG relative importance (%)")
    axes[1].set_title("Relative Importance (LMG)")
    for bar in bars:
        axes[1].text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2, f"{bar.get_width():.1f}%", va="center")
    axes[1].set_xlim(0, max(imps) * 1.22 if max(imps) > 0 else 1)
    fig.suptitle(f"OLS + LMG | {model_type} | Init {init} | {stg}", fontsize=14)
    fig.tight_layout()
    fig.savefig(filename, dpi=220, bbox_inches="tight")
    plt.close(fig)

for _, r in df_raw_ols.iterrows():
    plot_ols_lmg_combo(r, "#5B8FF9", f"{SUB_DIRS['ols_lmg_combo']}/ols_lmg_raw_{r['init_date']}_{r['stage']}_SHF.png")
for _, r in df_main_ols.iterrows():
    plot_ols_lmg_combo(r, "#2E8B57", f"{SUB_DIRS['ols_lmg_combo']}/ols_lmg_residualized_{r['init_date']}_{r['stage']}_SHF.png")
for _, r in df_sens_ols.iterrows():
    plot_ols_lmg_combo(r, "#E68600", f"{SUB_DIRS['ols_lmg_combo']}/ols_lmg_sensitivity_TPparam_{r['init_date']}_{r['stage']}_SHF.png")

# Preserve standalone LMG bar outputs for backward-compatible inspection.
for _, r in df_main_ols.iterrows():
    lmg_dict = parse_lmg_dict(r["lmg_dict"])
    if not lmg_dict:
        continue
    facs, imps = zip(*sorted(lmg_dict.items(), key=lambda item: item[1]))
    fig, ax = plt.subplots(figsize=(7, 5))
    bars = ax.barh([label_map.get(f, f) for f in facs], imps, color="steelblue", edgecolor="black")
    ax.set_title(f"LMG Relative Importance | Init {r['init_date']} | {r['stage']} | {resid_strategies.get(r['stage'], '')}", fontsize=13)
    ax.set_xlabel("Importance (%)")
    for bar in bars:
        ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2, f"{bar.get_width():.1f}%", va="center")
    ax.set_xlim(0, max(imps) * 1.22 if max(imps) > 0 else 1)
    fig.tight_layout()
    fig.savefig(f"{SUB_DIRS['lmg_main']}/lmg_{r['init_date']}_{r['stage']}_SHF.png", dpi=200, bbox_inches="tight")
    plt.close(fig)

for _, r in df_sens_ols.iterrows():
    lmg_dict = parse_lmg_dict(r["lmg_dict"])
    if not lmg_dict:
        continue
    facs, imps = zip(*sorted(lmg_dict.items(), key=lambda item: item[1]))
    fig, ax = plt.subplots(figsize=(7, 5))
    bars = ax.barh([label_map.get(f, f) for f in facs], imps, color="darkorange", edgecolor="black")
    ax.set_title(f"Sensitivity LMG | Init {r['init_date']} | {r['stage']} | TP_param_resid", fontsize=13)
    ax.set_xlabel("Importance (%)")
    for bar in bars:
        ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2, f"{bar.get_width():.1f}%", va="center")
    ax.set_xlim(0, max(imps) * 1.22 if max(imps) > 0 else 1)
    fig.tight_layout()
    fig.savefig(f"{SUB_DIRS['lmg_sensitivity']}/sens_lmg_{r['init_date']}_{r['stage']}_SHF.png", dpi=200, bbox_inches="tight")
    plt.close(fig)

# 3. Tmax stage metrics, time-series plots, and heatmaps.
metrics_records = []
daily_ols_projection_records = []
daily_projection_consistency_records = []
daily_projection_setup_records = []
daily_stage_predictor_compare_records = []
daily_projection_inconsistent = False
if not df_daily_tmax.empty:
    df_daily_tmax["date"] = pd.to_datetime(df_daily_tmax["date"])
    for init in start_date_list_run:
        init_daily = df_daily_tmax[df_daily_tmax["init_date"] == init]
        daily_stats = init_daily.groupby("date").agg(
            S2S_mean_absT=("s2s_tmax", "mean"),
            Spread_S2S=("s2s_tmax", "std"),
            ERA5_absT=("era5_tmax", "mean"),
        ).reset_index()
        stage_mean_rows_for_init = []
        for stage, (w_start, w_end) in study_window_dict_run.items():
            fit_group = df_fitted_all[(df_fitted_all["init_date"] == init) & (df_fitted_all["stage"] == stage) & (df_fitted_all["model_type"] == "residualized_main")]
            if fit_group.empty:
                continue
            mask = (daily_stats["date"] >= pd.Timestamp(w_start)) & (daily_stats["date"] <= pd.Timestamp(w_end))
            era5_vals = daily_stats.loc[mask, "ERA5_absT"].values
            s2s_vals = daily_stats.loc[mask, "S2S_mean_absT"].values
            ols_mean = float(fit_group["fitted_value"].mean())
            ols_spread = float(fit_group["fitted_value"].std())
            r2_match = df_main_ols[(df_main_ols["init_date"] == init) & (df_main_ols["stage"] == stage)]
            r2_member_axis = float(r2_match.iloc[0]["R2"]) if not r2_match.empty else np.nan
            metric_record = {
                "init_date": init,
                "stage": stage,
                "ERA5_absT": float(np.nanmean(era5_vals)),
                "S2S_mean_absT": float(np.nanmean(s2s_vals)),
                "OLS_stage_mean_absT": ols_mean,
                "Bias_S2S_vs_ERA5": float(np.nanmean(s2s_vals - era5_vals)),
                "RMSE_S2S_vs_ERA5": float(np.sqrt(np.nanmean((s2s_vals - era5_vals) ** 2))),
                # R2_S2S_vs_ERA5 is computed along the daily time axis.
                "R2_S2S_vs_ERA5": safe_corr_r2(s2s_vals, era5_vals),
                # OLS stage-mean RMSE is a stage-level external consistency diagnostic, not a daily forecast skill score.
                "Bias_OLS_stage_mean_vs_ERA5": float(np.nanmean(ols_mean - era5_vals)),
                "RMSE_OLS_stage_mean_vs_ERA5": float(np.sqrt(np.nanmean((ols_mean - era5_vals) ** 2))),
                "Spread_S2S": float(np.nanmean(daily_stats.loc[mask, "Spread_S2S"].values)),
                "Spread_OLS_member_fitted": ols_spread,
                # R2_OLS_member_axis is computed along the ensemble-member axis.
                # It should not be interpreted as the same type of skill score as R2_S2S_vs_ERA5.
                "R2_OLS_member_axis": r2_member_axis,
            }
            metrics_records.append(metric_record)
            stage_mean_rows_for_init.append(metric_record)
        ols_daily_projection, setup_debug, predictor_compare = build_daily_ols_projection(init, daily_stats, stage_mean_rows_for_init)
        if not setup_debug.empty:
            daily_projection_setup_records.extend(setup_debug.to_dict("records"))
        if not predictor_compare.empty:
            daily_stage_predictor_compare_records.extend(predictor_compare.to_dict("records"))
        if not ols_daily_projection.empty:
            daily_ols_projection_records.extend(ols_daily_projection.to_dict("records"))

def plot_stage_mean_diagnostic_only(init, reason_text):
    init_daily = df_daily_tmax[df_daily_tmax["init_date"] == init]
    daily_stats = init_daily.groupby("date").agg(
        S2S_mean_absT=("s2s_tmax", "mean"),
        Spread_S2S=("s2s_tmax", "std"),
        ERA5_absT=("era5_tmax", "mean"),
    ).reset_index()
    stage_mean_rows = []
    for rec in metrics_records:
        if rec["init_date"] != init or rec["stage"] not in DAILY_PROJECTION_STAGES:
            continue
        w_start, w_end = study_window_dict_run[rec["stage"]]
        for d in daily_stats.loc[(daily_stats["date"] >= pd.Timestamp(w_start)) & (daily_stats["date"] <= pd.Timestamp(w_end)), "date"]:
            stage_mean_rows.append({"date": d, "OLS_stage_mean_absT": rec["OLS_stage_mean_absT"], "stage": rec["stage"]})
    stage_mean_df = pd.DataFrame(stage_mean_rows)
    if stage_mean_df.empty:
        print(f"  [Warning] skipped stage-mean diagnostic plot for {init}: no stage mean rows")
        return
    plot_df = daily_stats.merge(stage_mean_df, on="date", how="left")
    fig, ax = plt.subplots(figsize=(11, 5.5))
    ax.axvspan(pd.Timestamp("2023-06-14"), pd.Timestamp("2023-06-17") + pd.Timedelta(hours=18), color="#F6D365", alpha=0.22, label="Stage-I_dry")
    ax.axvspan(pd.Timestamp("2023-06-17") + pd.Timedelta(hours=18), pd.Timestamp("2023-06-18") + pd.Timedelta(hours=12), color="#CCCCCC", alpha=0.22, label="transition / break")
    ax.axvspan(pd.Timestamp("2023-06-18") + pd.Timedelta(hours=12), pd.Timestamp("2023-06-24") + pd.Timedelta(hours=18), color="#A1C4FD", alpha=0.22, label="Stage-II_wet")
    ax.plot(plot_df["date"], plot_df["ERA5_absT"], color="black", marker="o", lw=2.0, label="ERA5 Obs")
    ax.plot(plot_df["date"], plot_df["S2S_mean_absT"], color="#5B8FF9", marker="o", lw=1.8, label="S2S ensemble mean")
    ax.fill_between(plot_df["date"], plot_df["S2S_mean_absT"] - plot_df["Spread_S2S"], plot_df["S2S_mean_absT"] + plot_df["Spread_S2S"], color="#5B8FF9", alpha=0.18, label="S2S ±1 std")
    ax.step(plot_df["date"], plot_df["OLS_stage_mean_absT"], where="mid", color="#2E8B57", lw=1.8, linestyle="--", label="OLS stage-mean diagnostic only")
    ax.text(0.01, 0.98, reason_text, transform=ax.transAxes, ha="left", va="top", bbox=dict(facecolor="white", alpha=0.75, edgecolor="none"))
    ax.set_title(f"Tmax Time Series | Stage-mean Diagnostic Only | Init {init}")
    ax.set_ylabel("Tmax (°C)")
    ax.set_xlabel("Date")
    ax.legend(loc="best", ncol=2, fontsize=9)
    fig.autofmt_xdate()
    fig.tight_layout()
    fig.savefig(f"{SUB_DIRS['tmax_timeseries']}/tmax_timeseries_stage_mean_diagnostic_{init}_SHF.png", dpi=220, bbox_inches="tight")
    plt.close(fig)

def plot_validated_daily_projection_timeseries(df_daily_projection_checked):
    for init in start_date_list_run:
        init_daily = df_daily_tmax[df_daily_tmax["init_date"] == init]
        daily_stats = init_daily.groupby("date").agg(
            S2S_mean_absT=("s2s_tmax", "mean"),
            Spread_S2S=("s2s_tmax", "std"),
            ERA5_absT=("era5_tmax", "mean"),
        ).reset_index()
        init_projection = df_daily_projection_checked[df_daily_projection_checked["init_date"] == init]
        plot_df = daily_stats.merge(init_projection[["date", "stage", "OLS_daily_projection", "OLS_daily_member_spread", "OLS_stage_mean_absT"]], on="date", how="left")
        fig, ax = plt.subplots(figsize=(11, 5.5))
        ax.axvspan(pd.Timestamp("2023-06-14"), pd.Timestamp("2023-06-17") + pd.Timedelta(hours=18), color="#F6D365", alpha=0.22, label="Stage-I_dry")
        ax.axvspan(pd.Timestamp("2023-06-17") + pd.Timedelta(hours=18), pd.Timestamp("2023-06-18") + pd.Timedelta(hours=12), color="#CCCCCC", alpha=0.22, label="transition / break")
        ax.axvspan(pd.Timestamp("2023-06-18") + pd.Timedelta(hours=12), pd.Timestamp("2023-06-24") + pd.Timedelta(hours=18), color="#A1C4FD", alpha=0.22, label="Stage-II_wet")
        ax.plot(plot_df["date"], plot_df["ERA5_absT"], color="black", marker="o", lw=2.0, label="ERA5 Obs")
        ax.plot(plot_df["date"], plot_df["S2S_mean_absT"], color="#5B8FF9", marker="o", lw=1.8, label="S2S ensemble mean")
        ax.fill_between(plot_df["date"], plot_df["S2S_mean_absT"] - plot_df["Spread_S2S"], plot_df["S2S_mean_absT"] + plot_df["Spread_S2S"], color="#5B8FF9", alpha=0.18, label="S2S ±1 std")
        ax.plot(plot_df["date"], plot_df["OLS_daily_projection"], color="#2E8B57", marker="s", lw=1.8, label="Residualized OLS daily projection")
        ax.fill_between(plot_df["date"], plot_df["OLS_daily_projection"] - plot_df["OLS_daily_member_spread"], plot_df["OLS_daily_projection"] + plot_df["OLS_daily_member_spread"], color="#2E8B57", alpha=0.12, label="OLS daily member spread")
        ax.step(plot_df["date"], plot_df["OLS_stage_mean_absT"], where="mid", color="#2E8B57", lw=1.1, alpha=0.45, linestyle="--", label="OLS stage-mean diagnostic")
        total_metrics = next((m for m in metrics_records if m["init_date"] == init and m["stage"] == "Total"), None)
        if total_metrics:
            ax.text(0.01, 0.98, f"OLS stage mean vs ERA5 Total: Bias={total_metrics['Bias_OLS_stage_mean_vs_ERA5']:.2f}°C | RMSE={total_metrics['RMSE_OLS_stage_mean_vs_ERA5']:.2f}°C | member-axis R²={total_metrics['R2_OLS_member_axis']:.2f}\nOLS stage-mean RMSE is a stage-level external diagnostic, not daily forecast skill.", transform=ax.transAxes, ha="left", va="top", bbox=dict(facecolor="white", alpha=0.75, edgecolor="none"))
        ax.set_title(f"Tmax Time Series | Init {init}")
        ax.set_ylabel("Tmax (°C)")
        ax.set_xlabel("Date")
        ax.legend(loc="best", ncol=2, fontsize=9)
        fig.autofmt_xdate()
        fig.tight_layout()
        fig.savefig(f"{SUB_DIRS['tmax_timeseries']}/tmax_timeseries_{init}_SHF.png", dpi=220, bbox_inches="tight")
        plt.close(fig)

expected_daily_projection_pairs = {(init, stage) for init in start_date_list_run for stage in DAILY_PROJECTION_STAGES}

if daily_projection_setup_records:
    df_projection_setup_out = pd.DataFrame(daily_projection_setup_records)
    setup_pairs = set(zip(df_projection_setup_out["init_date"], df_projection_setup_out["stage"]))
    missing_setup_pairs = expected_daily_projection_pairs - setup_pairs
    if missing_setup_pairs:
        raise RuntimeError(f"daily projection model setup missing expected init/stage pairs: {sorted(missing_setup_pairs)}")
    df_projection_setup_out.to_csv(daily_projection_setup_csv, index=False)
else:
    raise RuntimeError("daily projection model setup table is empty; cannot validate Stage-I/Stage-II projection setup")

if daily_stage_predictor_compare_records:
    df_predictor_compare_out = pd.DataFrame(daily_stage_predictor_compare_records)
    compare_pairs = set(zip(df_predictor_compare_out["init_date"], df_predictor_compare_out["stage"]))
    missing_compare_pairs = expected_daily_projection_pairs - compare_pairs
    if missing_compare_pairs:
        raise RuntimeError(f"daily-to-stage predictor comparison missing expected init/stage pairs: {sorted(missing_compare_pairs)}")
    df_predictor_compare_out.to_csv(daily_stage_predictor_compare_csv, index=False)
    predictor_diff_threshold = 1e-3
    predictor_warning_threshold = 1e-5
    warning_predictor_diffs = df_predictor_compare_out[(df_predictor_compare_out["max_abs_diff"].abs() > predictor_warning_threshold) & (df_predictor_compare_out["max_abs_diff"].abs() <= predictor_diff_threshold)]
    if not warning_predictor_diffs.empty:
        warning_details = warning_predictor_diffs[["init_date", "stage", "predictor_space", "factor", "max_abs_diff"]].to_dict("records")
        for rec in warning_details:
            print(f"  [Warning] daily-to-stage predictor mismatch within tolerance: init={rec['init_date']}, stage={rec['stage']}, predictor_space={rec['predictor_space']}, factor={rec['factor']}, max_abs_diff={rec['max_abs_diff']}")
    bad_predictor_diffs = df_predictor_compare_out[df_predictor_compare_out["max_abs_diff"].abs() > predictor_diff_threshold]
    if not bad_predictor_diffs.empty:
        details = bad_predictor_diffs[["init_date", "stage", "predictor_space", "factor", "max_abs_diff"]].to_dict("records")
        for rec in details:
            print(f"  [Error] daily-to-stage predictor mismatch: init={rec['init_date']}, stage={rec['stage']}, predictor_space={rec['predictor_space']}, factor={rec['factor']}, max_abs_diff={rec['max_abs_diff']}")
        raise RuntimeError(f"daily-to-stage predictor comparison exceeded threshold {predictor_diff_threshold}: {details}")
else:
    raise RuntimeError("daily-to-stage predictor comparison table is empty; cannot validate daily predictors against stage predictors")

if daily_ols_projection_records:
    df_daily_projection_out = pd.DataFrame(daily_ols_projection_records).sort_values(["init_date", "date", "stage"]).reset_index(drop=True)
    df_daily_projection_out.to_csv(daily_ols_projection_csv, index=False)
else:
    raise RuntimeError("daily OLS projection table was not generated; check table_daily_member_raw_factors_SHF.csv availability")

# Re-read the final saved projection table before all final row-count and consistency checks.
df_daily_projection_check = pd.read_csv(daily_ols_projection_csv)
df_daily_projection_check["date"] = pd.to_datetime(df_daily_projection_check["date"])
if len(df_daily_projection_check) != EXPECTED_DAILY_PROJECTION_ROWS:
    raise RuntimeError(f"daily OLS projection row check failed: expected {EXPECTED_DAILY_PROJECTION_ROWS}, got {len(df_daily_projection_check)}; file={daily_ols_projection_csv}")
projection_pairs = set(zip(df_daily_projection_check["init_date"], df_daily_projection_check["stage"]))
missing_projection_pairs = expected_daily_projection_pairs - projection_pairs
if missing_projection_pairs:
    raise RuntimeError(f"daily OLS projection missing expected init/stage pairs: {sorted(missing_projection_pairs)}")
for (init, stage), group in df_daily_projection_check.groupby(["init_date", "stage"]):
    expected_days = EXPECTED_DAILY_PROJECTION_STAGE_DAYS[stage]
    actual_days = group["date"].nunique()
    if actual_days != expected_days:
        raise RuntimeError(f"daily OLS projection day-count check failed for {init}/{stage}: expected {expected_days}, got {actual_days}")

# Consistency is intentionally computed from the final saved CSV, not from an in-memory pre-write object.
daily_projection_consistency_records = []
daily_projection_inconsistent = False
for (init, stage), group in df_daily_projection_check.groupby(["init_date", "stage"], sort=False):
    stage_mean_absT = float(group["OLS_stage_mean_absT"].iloc[0])
    daily_mean = float(group["OLS_daily_projection"].mean())
    if not np.isfinite(daily_mean) or not np.isfinite(stage_mean_absT):
        raise RuntimeError(f"daily OLS projection finite check failed for {init}/{stage}: mean_daily={daily_mean}, OLS_stage_mean_absT={stage_mean_absT}")
    diff = daily_mean - stage_mean_absT
    warn_flag = abs(diff) > 0.05
    daily_projection_inconsistent = daily_projection_inconsistent or warn_flag
    daily_projection_consistency_records.append({
        "init_date": init,
        "stage": stage,
        "n_days": int(group["date"].nunique()),
        "mean_over_days_OLS_daily_projection": daily_mean,
        "OLS_stage_mean_absT": stage_mean_absT,
        "difference": diff,
        "abs_difference": abs(diff),
        "warning_abs_diff_gt_0p05C": warn_flag,
        "checked_from_final_saved_csv": True,
    })
pd.DataFrame(daily_projection_consistency_records).to_csv(daily_projection_consistency_csv, index=False)
bad_consistency = [rec for rec in daily_projection_consistency_records if rec["warning_abs_diff_gt_0p05C"]]
if bad_consistency:
    reason = "Daily OLS projection failed the ±0.05°C consistency check; this robust figure shows only the stage-mean diagnostic."
    for init in start_date_list_run:
        plot_stage_mean_diagnostic_only(init, reason)
    for rec in bad_consistency:
        print(f"  [Warning] Daily OLS projection consistency check failed for {rec['init_date']} | {rec['stage']}: mean_daily={rec['mean_over_days_OLS_daily_projection']:.3f}, stage_mean={rec['OLS_stage_mean_absT']:.3f}, diff={rec['difference']:.3f}°C")
    raise RuntimeError(f"daily OLS projection consistency failed after re-reading final CSV: {bad_consistency}")

# All final CSV checks passed; only now write the formal Tmax time-series figures.
plot_validated_daily_projection_timeseries(df_daily_projection_check)

df_metrics = pd.DataFrame(metrics_records)
if not df_metrics.empty:
    for bad_col in ["R2_OLS_vs_ERA5", "Bias_OLS_vs_ERA5", "RMSE_OLS_vs_ERA5", "OLS_mean_absT", "Spread_OLS"]:
        assert bad_col not in df_metrics.columns, f"Deprecated metric column still exists: {bad_col}"
    for good_col in [
        "OLS_stage_mean_absT",
        "Bias_OLS_stage_mean_vs_ERA5",
        "RMSE_OLS_stage_mean_vs_ERA5",
        "Spread_OLS_member_fitted",
        "R2_OLS_member_axis",
    ]:
        assert good_col in df_metrics.columns, f"Expected metric column missing: {good_col}"

    df_metrics.to_csv(f"{SUB_DIRS['tables']}/table_tmax_stage_metrics_SHF.csv", index=False)
    df_metrics["init_stage"] = df_metrics["init_date"].str.replace("2023-06-", "06", regex=False) + "_" + df_metrics["stage"].map(clean_stage_name)
    heatmap_specs = [
        ("absolute_temperature_heatmap_SHF.png", ["ERA5_absT", "S2S_mean_absT", "OLS_stage_mean_absT"], "Absolute Temperature Heatmap (°C)", "YlOrRd"),
        ("error_metrics_heatmap_SHF.png", ["Bias_S2S_vs_ERA5", "RMSE_S2S_vs_ERA5", "Bias_OLS_stage_mean_vs_ERA5", "RMSE_OLS_stage_mean_vs_ERA5"], "Error Metrics Heatmap (external bias/RMSE only; OLS stage-mean RMSE is not daily forecast skill)", "coolwarm"),
        ("spread_heatmap_SHF.png", ["Spread_S2S", "Spread_OLS_member_fitted"], "Spread Heatmap (member std)", "viridis"),
        ("ols_member_axis_r2_heatmap_SHF.png", ["R2_OLS_member_axis"], "OLS Fitting Quality Heatmap (member-axis R²)", "YlGnBu"),
        ("s2s_daily_tracking_r2_heatmap_SHF.png", ["R2_S2S_vs_ERA5"], "S2S Daily Tracking Heatmap (daily time-axis R²)", "PuBuGn"),
    ]
    for fname, cols, title, cmap in heatmap_specs:
        plot_mat = df_metrics.set_index("init_stage")[cols]
        fig_h = max(4, 0.55 * len(plot_mat) + 1.5)
        fig, ax = plt.subplots(figsize=(max(7, 1.4 * len(cols)), fig_h))
        sns.heatmap(plot_mat, annot=True, fmt=".2f", cmap=cmap, ax=ax, linewidths=0.5, linecolor="white")
        ax.set_title(f"{title}")
        ax.set_xlabel("")
        ax.set_ylabel("init_stage")
        fig.tight_layout()
        fig.savefig(f"{SUB_DIRS['metrics_heatmap']}/{fname}", dpi=220, bbox_inches="tight")
        plt.close(fig)

    summary_rows = []
    stage_order = ["Stage-I_dry", "Stage-II_wet", "Total"]
    era5_stage = df_metrics.drop_duplicates("stage").set_index("stage")
    for stage in stage_order:
        if stage in era5_stage.index:
            summary_rows.append({"block": "ERA5 observation", "row_metric": "ERA5_absT", "stage": stage, "value": float(era5_stage.loc[stage, "ERA5_absT"]), "unit": "degC"})
    for init in start_date_list_run:
        init_metrics = df_metrics[df_metrics["init_date"] == init].set_index("stage")
        for metric in ["S2S_mean_absT", "Bias_S2S_vs_ERA5", "RMSE_S2S_vs_ERA5", "Spread_S2S", "OLS_stage_mean_absT", "Bias_OLS_stage_mean_vs_ERA5", "RMSE_OLS_stage_mean_vs_ERA5", "Spread_OLS_member_fitted"]:
            for stage in stage_order:
                if stage in init_metrics.index:
                    summary_rows.append({"block": f"Init {init}", "row_metric": metric, "stage": stage, "value": float(init_metrics.loc[stage, metric]), "unit": "degC"})
    df_abs_error_summary = pd.DataFrame(summary_rows)
    df_abs_error_summary.to_csv(f"{SUB_DIRS['tables']}/table_tmax_abs_error_summary.csv", index=False)

    row_keys = df_abs_error_summary[["block", "row_metric"]].drop_duplicates().itertuples(index=False, name=None)
    row_keys = list(row_keys)
    pivot = df_abs_error_summary.pivot_table(index=["block", "row_metric"], columns="stage", values="value", aggfunc="first").reindex(row_keys)
    pivot = pivot.reindex(columns=stage_order)
    all_bias = df_abs_error_summary[df_abs_error_summary["row_metric"].str.contains("Bias")]["value"].abs().values
    all_rmse = df_abs_error_summary[df_abs_error_summary["row_metric"].str.contains("RMSE")]["value"].values
    all_spread = df_abs_error_summary[df_abs_error_summary["row_metric"].str.contains("Spread")]["value"].values
    bias_lim = max(float(np.nanpercentile(all_bias[np.isfinite(all_bias)], 95)) if np.isfinite(all_bias).any() else 1.0, 0.1)
    rmse_max = max(float(np.nanpercentile(all_rmse[np.isfinite(all_rmse)], 95)) if np.isfinite(all_rmse).any() else 1.0, 0.1)
    spread_max = max(float(np.nanpercentile(all_spread[np.isfinite(all_spread)], 95)) if np.isfinite(all_spread).any() else 1.0, 0.1)
    bias_norm = mcolors.TwoSlopeNorm(vmin=-bias_lim, vcenter=0.0, vmax=bias_lim)
    rmse_norm = mcolors.Normalize(vmin=0.0, vmax=rmse_max)
    spread_norm = mcolors.Normalize(vmin=0.0, vmax=spread_max)
    # Truncated + whitened colormaps keep the table paper-like: color supports interpretation without overpowering numbers.
    bias_cmap = truncate_colormap("RdBu_r", 0.20, 0.80)
    rmse_cmap = truncate_colormap("Oranges", 0.05, 0.45)
    spread_cmap = truncate_colormap("BuGn", 0.05, 0.45)
    abs_temp_facecolor = "#fbfbfb"
    bias_lighten = 0.62
    rmse_lighten = 0.58
    spread_lighten = 0.58

    fig, ax = plt.subplots(figsize=(9.5, max(5.5, 0.42 * len(pivot) + 2.0)))
    ax.set_xlim(-2.7, len(stage_order))
    ax.set_ylim(len(pivot), -1.2)
    ax.axis("off")
    for j, stage in enumerate(stage_order):
        ax.text(j + 0.5, -0.35, stage, ha="center", va="center", fontweight="bold")
    last_block = None
    for i, ((block, metric), row) in enumerate(pivot.iterrows()):
        if block != last_block:
            ax.text(-2.62, i + 0.5, block, ha="left", va="center", fontweight="bold")
            last_block = block
        ax.text(-0.2, i + 0.5, metric, ha="right", va="center")
        for j, stage in enumerate(stage_order):
            val = row.get(stage, np.nan)
            if not np.isfinite(val):
                facecolor = "white"
                label = ""
            elif "Bias" in metric:
                facecolor = lighten_color(bias_cmap(bias_norm(np.clip(val, -bias_lim, bias_lim))), amount=bias_lighten)
                label = f"{val:.2f}°C"
            elif "RMSE" in metric:
                facecolor = lighten_color(rmse_cmap(rmse_norm(np.clip(max(val, 0.0), 0.0, rmse_max))), amount=rmse_lighten)
                label = f"{val:.2f}°C"
            elif "Spread" in metric:
                facecolor = lighten_color(spread_cmap(spread_norm(np.clip(max(val, 0.0), 0.0, spread_max))), amount=spread_lighten)
                label = f"{val:.2f}°C"
            else:
                facecolor = abs_temp_facecolor
                label = f"{val:.2f}°C"
            ax.add_patch(patches.Rectangle((j, i), 1, 1, facecolor=facecolor, edgecolor="#e0e0e0", linewidth=1.0))
            ax.text(j + 0.5, i + 0.5, label, ha="center", va="center", fontsize=10, color=readable_text_color(facecolor))
    ax.set_title("Tmax Absolute Temperature and Error Summary", fontsize=14, pad=20)
    ax.text(0.5, 1.01, "Absolute temperature rows are unshaded; Bias, RMSE, and Spread use separate visual encodings.", transform=ax.transAxes, ha="center", va="bottom", fontsize=10)
    fig.tight_layout()
    fig.savefig(f"{SUB_DIRS['metrics_heatmap']}/fig_tmax_abs_error_summary_table.png", dpi=240, bbox_inches="tight")
    plt.close(fig)
else:
    print("  [Warning] table_daily_tmax_SHF.csv is missing/empty; skipped Tmax time-series and metric heatmaps.")

print("\n" + "=" * 80)
print("【终端 Summary】")
print(f"ERA5 Tmax file path: {ERA5_TMAX_FILE}")
if daily_projection_setup_records:
    print("[Daily OLS projection setup check]")
    for rec in daily_projection_setup_records:
        print(f"  {rec['init_date']} | {rec['stage']}: coefficients={rec['same_ols_coefficients_as_stage_model']}, intercept={rec['same_intercept_as_stage_model']}, scaler={rec['same_scaler_as_stage_model']}, residualization={rec['same_residualization_equations_as_stage_model']}, SHF/sign={rec['same_factor_sign_convention']}, double_baseline={rec['baseline_double_added']}")
if daily_projection_consistency_records:
    print("[Daily OLS projection consistency]")
    for rec in daily_projection_consistency_records:
        status = "WARNING" if rec["warning_abs_diff_gt_0p05C"] else "OK"
        print(f"  {rec['init_date']} | {rec['stage']}: mean_daily={rec['mean_over_days_OLS_daily_projection']:.3f}, OLS_stage_mean={rec['OLS_stage_mean_absT']:.3f}, diff={rec['difference']:.3f}°C [{status}]")
if daily_stage_predictor_compare_records:
    max_diff = max(abs(rec["max_abs_diff"]) for rec in daily_stage_predictor_compare_records)
    print(f"[Daily→stage predictor comparison] max_abs_diff={max_diff:.6g}; tolerance=1e-3; details: {daily_stage_predictor_compare_csv}")
for _, r in df_main_ols.iterrows():
    init = r["init_date"]
    stg = r["stage"]
    lmg = parse_lmg_dict(r["lmg_dict"])
    print(f"[{init} | {stg}]")
    print(f"  OLS main model: R²={r['R2']:.2f}, Adj-R²={r['Adj_R2']:.2f}, Top1={r['Top1_factor_name']} ({r['Top1_factor_importance_pct']:.1f}%)")
    if stg == "Stage-I_dry":
        sum_imp = lmg.get("sm_avg", 0) + lmg.get("SHF_resid", 0) + lmg.get("SSR_resid", 0)
        print(f"  [Coupling Impt] sm_avg + SHF_resid + SSR_resid = {sum_imp:.1f}%")
    else:
        sum_imp = lmg.get("NCHN_tp", 0) + lmg.get("SM_resid", 0) + lmg.get("SSR_resid", 0) + lmg.get("SHF_resid", 0)
        print(f"  [Coupling Impt] NCHN_tp + SM_resid + SSR_resid + SHF_resid = {sum_imp:.1f}%")
    if not df_metrics.empty:
        mm = df_metrics[(df_metrics["init_date"] == init) & (df_metrics["stage"] == stg)]
        if not mm.empty:
            m = mm.iloc[0]
            print(f"  S2S vs ERA5: Bias={m['Bias_S2S_vs_ERA5']:.2f}, RMSE={m['RMSE_S2S_vs_ERA5']:.2f}, daily time-axis R²={m['R2_S2S_vs_ERA5']:.2f}, Spread={m['Spread_S2S']:.2f}")
            print(f"  OLS stage mean vs ERA5: Bias={m['Bias_OLS_stage_mean_vs_ERA5']:.2f}, RMSE={m['RMSE_OLS_stage_mean_vs_ERA5']:.2f}, Spread={m['Spread_OLS_member_fitted']:.2f}")
            print(f"  OLS member-axis fitting: R²={m['R2_OLS_member_axis']:.2f}")
    if not df_factor_direction.empty:
        diag = df_factor_direction[(df_factor_direction["init_date"] == init) & (df_factor_direction["stage"] == stg)]
        diag = diag[diag["factor"].isin(["SHF_Avg", "SHF_resid", "sm_avg", "SSR_resid", "z500_anom_NCHN"])]
        if not diag.empty:
            print("  Factor direction diagnostics (member-axis r / standardized coef):")
            for _, drow in diag.iterrows():
                factor_label = "SHF_Avg(up)" if drow["factor"] == "SHF_Avg" else drow["factor"]
                print(f"    [{drow['model_kind']}] {factor_label}: r={drow['pearson_r_with_y']:.2f}, beta_std={drow['ols_coef_standardized']:.2f}")
        neg_shf = diag[(diag["model_kind"] == "residualized_main") & (diag["factor"] == "SHF_resid") & (diag["pearson_r_with_y"] < 0) & (diag["ols_coef_standardized"] < 0)]
        if stg == "Stage-I_dry" and not neg_shf.empty:
            print("  SHF_resid shows negative member-axis association with Tmax; this is a physical/statistical sign, not a plotting error.")
    if stg == "Stage-II_wet" and not df_sens_ols.empty:
        sens_match = df_sens_ols[(df_sens_ols["init_date"] == init) & (df_sens_ols["stage"] == stg)]
        if not sens_match.empty:
            sens_lmg = parse_lmg_dict(sens_match.iloc[0]["lmg_dict"])
            print(f"  [Sensitivity] TP_param_resid Importance = {sens_lmg.get('TP_param_resid', 0):.1f}%")
print("=" * 80)
print("SHF sign convention: SHF_Avg was converted in Cell 1 and stored as upward-positive; Cell 2/3 do not flip it again.")
print(f"Stage-Specific Residualized OLS (SHF Version) 执行完毕！结果位于: {RESID_OLS_DIR}")
print("=" * 80)
